# Martingale-DML Unified Simulation Notebook (Simplified)

This version is fully self-contained and simplifies controls:
- Use `N_REPS` directly (no `fast/paper` mode).
- Produce plain CSV summary tables for manual Overleaf work.
- No LaTeX table generation required.


In [15]:
#!/usr/bin/env python
# coding: utf-8

# # Martingale-DML Unified Simulation Notebook
# 
# This notebook is a **self-contained** implementation of your simulation codebase.
# It inlines all code needed to run simulations for **Designs A, B, C1, C2, D** and the
# standalone **regime-switching studentization** experiment, without importing local modules
# like `sim/*`.
# 
# ## What this notebook does
# - Defines all configs, estimators, nuisance learners, design generators, and runners inline.
# - Runs selected designs and writes replication artifacts to `artifacts/sim/{design}/{method}/replications.csv`.
# - Builds summary tables in `tables/`.
# - Builds figures in `figures/`.
# - Runs the regime-switching studentization experiment and writes its outputs.
# 
# ## Design C Split (Theory-Facing)
# - **Design C1 (Leakage / predictability stress test):** strong adaptive feedback in assignment, with a past-only (predictable) nuisance vs a leaky 5-fold nuisance that uses future outcomes. Intended to make Remark 4.4’s warning operational and to stress the MDS/predictability requirements behind Lemma 5.3 and Theorem 5.14 (Assumption 4.5; Remark 4.4; Lemma 5.3; Theorem 5.14).
# - **Design C2 (Oracle precision / nuisance-quality knob):** fixed propensity `pi_t ≡ 0.5` so nuisance quality is the only precision knob; compare IPW (m≡0) vs misspecified linear vs well-specified linear vs oracle, tying directly to the nonnegative augmentation term/variance inflation in Proposition 5.8 and the oracle-equivalence message in Theorem C.4 (Proposition 5.8; Theorem C.4; Theorem 5.14).
# 
# ## Dependency notes
# Install the same dependencies used by your repo:
# ```bash
# pip install -r requirements.txt -r requirements-sim.txt
# ```

# In[2]:


from __future__ import annotations

import json
import math
import zlib
from dataclasses import asdict, dataclass
from pathlib import Path
from statistics import NormalDist

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from joblib import Parallel, delayed
except Exception:
    def delayed(func):
        def _wrap(*args, **kwargs):
            return lambda: func(*args, **kwargs)
        return _wrap

    class Parallel:
        def __init__(self, n_jobs: int | None = None):
            self.n_jobs = n_jobs

        def __call__(self, tasks):
            return [task() for task in tasks]
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import KFold

try:
    from scipy.stats import norm as scipy_norm
    norm_ppf = scipy_norm.ppf
except Exception:
    norm_ppf = NormalDist().inv_cdf

np.set_printoptions(suppress=True)
pd.set_option("display.max_columns", 200)


# ## Core Config, RNG, Estimation, Nuisance Models, Metrics, Tables, Figures
# 
# The next cell combines the functionality previously split across:
# - `sim/config.py`
# - `sim/rng.py`
# - `sim/estimators.py`
# - `sim/baselines.py`
# - `sim/nuisances.py`
# - `sim/metrics.py`
# - `sim/tables.py`
# - `sim/plotting.py`

# In[ ]:


@dataclass(frozen=True)
class GlobalConfig:
    seed: int = 12345
    alpha: float = 0.05
    n_list: tuple[int, ...] = (250, 500)
    n_reps: int = 50
    n_jobs: int = -1
    artifacts_dir: Path = Path("artifacts/sim")
    tables_dir: Path = Path("tables")
    figures_dir: Path = Path("figures")

    def resolve_R(self) -> int:
        return int(self.n_reps)

    def resolve_n_list(self) -> list[int]:
        return list(self.n_list)


@dataclass(frozen=True)
class DesignAConfig:
    name: str = "A"
    n0: int = 50
    tau: float = 0.0
    sigma0: float = 1.0
    sigma1: float = 3.0
    pi_low: float = 0.2
    pi_high: float = 0.8
    pi_burn: float = 0.5
    fixed_pi: float = 0.5
    n_list_override: tuple[int, ...] | None = None

    def n_list(self, global_cfg: GlobalConfig) -> list[int]:
        return list(self.n_list_override) if self.n_list_override is not None else global_cfg.resolve_n_list()


@dataclass(frozen=True)
class DesignBConfig:
    name: str = "B"
    tau: float = 0.0
    sigma0: float = 1.0
    sigma1: float = 3.0
    pi: float = 0.6
    n_list_override: tuple[int, ...] | None = None

    def n_list(self, global_cfg: GlobalConfig) -> list[int]:
        return list(self.n_list_override) if self.n_list_override is not None else global_cfg.resolve_n_list()


@dataclass(frozen=True)
class DesignC1Config:
    # Leakage / predictability stress test (C1).
    name: str = "C1"
    # Covariates: X_t in R^p, with heterogeneous treatment effects whose mean is tau0 (so ATE=tau0).
    p: int = 20
    # Forward blocks (burn-in is block 0, score blocks 1..K-1). Keep consistent with the paper's
    # "forward" predictability pattern.
    K: int = 5
    # Calibration is reported at tau0=0; keep a small grid only if you also want power curves.
    tau_grid: tuple[float, ...] = (0.0,)
    delta: float = 0.2

    # Overlap (executed propensities are clipped to [eps, 1-eps]).
    eps: float = 0.1

    # Adaptive feedback strength: pi_t = eps + (1-2eps)*expit(lam * (m1_hat - m0_hat)).
    lam: float = 2.5

    # Nuisance fits:
    # - Predictable: low-dimensional feature map + ridge.
    # - Leaky: richer feature map fit on the full sample (uses future outcomes), intentionally violating predictability.
    ridge_pred: float = 0.1
    ridge_leaky: float = 1e-6

    # Heavy-tailed noise makes the stress test sharper while keeping moments finite.
    df: int = 30
    n_list_override: tuple[int, ...] | None = None

    def n_list(self, global_cfg: GlobalConfig) -> list[int]:
        return list(self.n_list_override) if self.n_list_override is not None else global_cfg.resolve_n_list()


@dataclass(frozen=True)
class DesignC2Config:
    # Oracle precision / nuisance-quality knob (C2).
    name: str = "C2"
    K: int = 10
    d: int = 5
    rho: float = 0.5
    pi: float = 0.5
    tau_grid: tuple[float, ...] = (0.0, 0.1, 0.2, 0.4)
    sigma: float = 1.0
    n_list_override: tuple[int, ...] | None = None

    def n_list(self, global_cfg: GlobalConfig) -> list[int]:
        return list(self.n_list_override) if self.n_list_override is not None else global_cfg.resolve_n_list()


@dataclass(frozen=True)
class DesignDConfig:
    name: str = "D"
    n0: int = 100
    block_size: int = 100
    d: int = 10
    rho: float = 0.3
    eps_greedy: float = 0.1
    softmax_temp: float = 0.5
    clip_low: float = 0.05
    clip_high: float = 0.95
    learner: str = "linear"
    n_list_override: tuple[int, ...] | None = None

    def n_list(self, global_cfg: GlobalConfig) -> list[int]:
        return list(self.n_list_override) if self.n_list_override is not None else global_cfg.resolve_n_list()


@dataclass(frozen=True)
class DesignEConfig:
    name: str = "E"
    n0: int = 200
    block_sizes: tuple[int, ...] = (25, 50, 100)
    d: int = 50
    rho: float = 0.2
    learners: tuple[str, ...] = ("linear", "lasso", "rf", "gbrt")
    clip_low: float = 0.05
    clip_high: float = 0.95
    n_list_override: tuple[int, ...] | None = None

    def n_list(self, global_cfg: GlobalConfig) -> list[int]:
        return list(self.n_list_override) if self.n_list_override is not None else global_cfg.resolve_n_list()


@dataclass(frozen=True)
class DesignFConfig:
    name: str = "F"
    n0: int = 100
    block_size: int = 50
    d: int = 10
    rho: float = 0.3
    clip_lows: tuple[float, ...] = (0.01, 0.05, 0.10)
    softmax_temp: float = 0.5
    learner: str = "gbrt"
    n_list_override: tuple[int, ...] | None = None
    heavy_df: tuple[int, ...] = (3, 5)

    def n_list(self, global_cfg: GlobalConfig) -> list[int]:
        return list(self.n_list_override) if self.n_list_override is not None else global_cfg.resolve_n_list()


DESIGN_IDS = {
    "A": 1,
    "B": 2,
    # Design C split into C1/C2.
    "C1": 31,
    "C2": 32,
    "D": 4,
    "E": 5,
    "F": 6,
}


def stable_hash(text: str) -> int:
    return zlib.crc32(text.encode("utf-8"))


@dataclass(frozen=True)
class ReplicationRng:
    rng: np.random.Generator
    seed_seq: np.random.SeedSequence

    def next_seed(self) -> int:
        return int(self.seed_seq.generate_state(1)[0])


def make_replication_rng(
    seed: int,
    design: str,
    n: int,
    rep: int,
    extra: str | None = None,
) -> ReplicationRng:
    parts: list[int] = [seed, DESIGN_IDS.get(design, 0), int(n), int(rep)]
    if extra is not None:
        parts.append(stable_hash(extra))
    seed_seq = np.random.SeedSequence(parts)
    return ReplicationRng(rng=np.random.default_rng(seed_seq), seed_seq=seed_seq)


@dataclass(frozen=True)
class EstimationResult:
    theta_hat: float
    vhat: float
    se: float
    ci_lower: float
    ci_upper: float
    n_eff: int

    @property
    def length(self) -> float:
        return self.ci_upper - self.ci_lower


@dataclass(frozen=True)
class FixedVarianceResult:
    theta_hat: float
    v_fix: float
    se: float
    ci_lower: float
    ci_upper: float
    n_eff: int

    @property
    def length(self) -> float:
        return self.ci_upper - self.ci_lower


def phi_score(
    y: np.ndarray,
    a: np.ndarray,
    pi: np.ndarray,
    m0: np.ndarray,
    m1: np.ndarray,
) -> np.ndarray:
    return (m1 - m0) + a * (y - m1) / pi - (1 - a) * (y - m0) / (1 - pi)


def theta_hat(phi: np.ndarray) -> float:
    return float(np.mean(phi))


def vhat(phi: np.ndarray) -> float:
    n_eff = phi.size
    if n_eff < 2:
        raise ValueError("n_eff must be at least 2 for Vhat.")
    centered = phi - np.mean(phi)
    return float(np.sum(centered ** 2) / (n_eff - 1))


def ci_critical(alpha: float, n_eff: int, critical: str = "normal") -> float:
    if critical == "normal":
        return float(stats.norm.ppf(1.0 - alpha / 2.0))
    if critical == "t":
        return float(stats.t.ppf(1.0 - alpha / 2.0, df=n_eff - 1))
    raise ValueError(f"Unknown critical '{critical}'.")


def compute_sn_ci(phi: np.ndarray, alpha: float, critical: str = "normal") -> EstimationResult:
    n_eff = int(phi.size)
    if n_eff < 2:
        raise ValueError("n_eff must be at least 2 for SN CI.")
    th = theta_hat(phi)
    vh = vhat(phi)
    se = math.sqrt(vh / n_eff)
    crit = ci_critical(alpha=alpha, n_eff=n_eff, critical=critical)
    return EstimationResult(
        theta_hat=th,
        vhat=vh,
        se=se,
        ci_lower=th - crit * se,
        ci_upper=th + crit * se,
        n_eff=n_eff,
    )


def winsorize(phi: np.ndarray, lower_q: float = 0.01, upper_q: float = 0.99) -> np.ndarray:
    if phi.size == 0:
        return phi
    lo = float(np.quantile(phi, lower_q))
    hi = float(np.quantile(phi, upper_q))
    return np.clip(phi, lo, hi)


def studentized_statistic(theta_hat_val: float, theta_true: float, se: float) -> float:
    if se <= 0:
        return float("nan")
    return float((theta_hat_val - theta_true) / se)


def fixed_variance_ci(
    theta_hat: float,
    v_fix: float,
    n_eff: int,
    alpha: float,
    critical: str = "normal",
) -> FixedVarianceResult:
    if n_eff < 1:
        raise ValueError("n_eff must be positive for fixed-variance CI.")
    if critical == "normal":
        crit = float(stats.norm.ppf(1.0 - alpha / 2.0))
    elif critical == "t":
        crit = float(stats.t.ppf(1.0 - alpha / 2.0, df=n_eff - 1))
    else:
        raise ValueError(f"Unknown critical '{critical}'.")
    se = math.sqrt(v_fix / n_eff)
    return FixedVarianceResult(
        theta_hat=theta_hat,
        v_fix=v_fix,
        se=se,
        ci_lower=theta_hat - crit * se,
        ci_upper=theta_hat + crit * se,
        n_eff=n_eff,
    )


def regime_variance_design_a(regime_high: bool) -> float:
    return 16.25 if regime_high else 46.25


class ConstantModel:
    def __init__(self, value: float):
        self.value = float(value)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.full(X.shape[0], self.value)


@dataclass(frozen=True)
class LearnerSpec:
    name: str
    random_state: int


def make_learner(spec: LearnerSpec):
    name = spec.name.lower()
    if name == "ols":
        return LinearRegression()
    if name == "linear":
        return Ridge(alpha=1e-3, random_state=spec.random_state)
    if name == "lasso":
        return Lasso(alpha=0.01, max_iter=5000, random_state=spec.random_state)
    if name == "rf":
        return RandomForestRegressor(
            n_estimators=200,
            max_depth=6,
            min_samples_leaf=5,
            random_state=spec.random_state,
        )
    if name == "gbrt":
        return GradientBoostingRegressor(
            n_estimators=120,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            random_state=spec.random_state,
        )
    raise ValueError(f"Unknown learner '{spec.name}'.")


def fit_arm_models(
    X: np.ndarray,
    A: np.ndarray,
    Y: np.ndarray,
    spec: LearnerSpec,
) -> tuple[object, object]:
    if X.shape[0] == 0:
        return ConstantModel(0.0), ConstantModel(0.0)

    idx0 = A == 0
    idx1 = A == 1

    if idx0.sum() == 0:
        model0 = ConstantModel(float(np.mean(Y)))
    else:
        model0 = make_learner(spec)
        model0.fit(X[idx0], Y[idx0])

    if idx1.sum() == 0:
        model1 = ConstantModel(float(np.mean(Y)))
    else:
        model1 = make_learner(spec)
        model1.fit(X[idx1], Y[idx1])

    return model0, model1


def predictable_block_predictions(
    X: np.ndarray,
    A: np.ndarray,
    Y: np.ndarray,
    blocks: list[np.ndarray],
    spec: LearnerSpec,
    include_current: bool = False,
) -> tuple[np.ndarray, np.ndarray]:
    n = X.shape[0]
    m0 = np.zeros(n)
    m1 = np.zeros(n)
    for k, block in enumerate(blocks):
        if include_current:
            train_idx = np.concatenate(blocks[: k + 1]) if k >= 0 else np.array([], dtype=int)
        else:
            train_idx = np.concatenate(blocks[:k]) if k > 0 else np.array([], dtype=int)
        if train_idx.size == 0:
            m0[block] = 0.0
            m1[block] = 0.0
            continue
        model0, model1 = fit_arm_models(X[train_idx], A[train_idx], Y[train_idx], spec)
        m0[block] = model0.predict(X[block])
        m1[block] = model1.predict(X[block])
    return m0, m1


def kfold_predictions(
    X: np.ndarray,
    A: np.ndarray,
    Y: np.ndarray,
    spec: LearnerSpec,
    n_splits: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    n = X.shape[0]
    m0 = np.zeros(n)
    m1 = np.zeros(n)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=spec.random_state)
    for train_idx, test_idx in kf.split(X):
        if train_idx.size == 0:
            continue
        model0, model1 = fit_arm_models(X[train_idx], A[train_idx], Y[train_idx], spec)
        m0[test_idx] = model0.predict(X[test_idx])
        m1[test_idx] = model1.predict(X[test_idx])
    return m0, m1


def mcse(p_hat: float, R: int) -> float:
    if R <= 0:
        return float("nan")
    return math.sqrt(max(p_hat * (1.0 - p_hat), 0.0) / R)


def add_basic_metrics(df: pd.DataFrame, theta_true: float) -> pd.DataFrame:
    df = df.copy()
    df["covered"] = (df["ci_lower"] <= theta_true) & (theta_true <= df["ci_upper"])
    df["length"] = df["ci_upper"] - df["ci_lower"]
    df["bias"] = df["theta_hat"] - theta_true
    df["sq_error"] = (df["theta_hat"] - theta_true) ** 2
    return df


def summarize(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    grouped = df.groupby(group_cols, dropna=False)
    summary = grouped.agg(
        coverage=("covered", "mean"),
        avg_len=("length", "mean"),
        bias=("bias", "mean"),
        rmse=("sq_error", lambda x: float(math.sqrt(np.mean(x)))),
        n_rep=("covered", "size"),
    ).reset_index()
    summary["mcse"] = summary.apply(lambda row: mcse(row["coverage"], int(row["n_rep"])), axis=1)
    return summary


def conditional_coverage_by_decile(
    df: pd.DataFrame,
    theta_true: float,
    q_col: str = "q_hat",
    method_col: str = "method",
    n_bins: int = 10,
) -> pd.DataFrame:
    df = add_basic_metrics(df, theta_true)
    df = df.copy()
    df["q_decile"] = pd.qcut(df[q_col], q=n_bins, labels=False, duplicates="drop")
    grouped = df.groupby([method_col, "q_decile"], dropna=False)
    summary = grouped.agg(
        coverage=("covered", "mean"),
        avg_len=("length", "mean"),
        n_rep=("covered", "size"),
        q_min=(q_col, "min"),
        q_max=(q_col, "max"),
    ).reset_index()
    summary["mcse"] = np.sqrt(summary["coverage"] * (1.0 - summary["coverage"]) / summary["n_rep"])
    return summary



def save_fig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig = plt.gcf()
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)



def plot_var_ratio_hist(values: np.ndarray, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(values, bins=30, color="#4C4C4C", edgecolor="black", alpha=0.8)
    ax.set_xlabel(r"$\widehat V$")
    ax.set_ylabel("Count")
    save_fig(path)


def plot_hist_qq(tstats: dict[str, np.ndarray], path: Path) -> None:
    n_rows = len(tstats)
    fig, axes = plt.subplots(n_rows, 2, figsize=(8, 3.5 * n_rows))
    if n_rows == 1:
        axes = np.array([axes])
    for row_idx, (label, vals) in enumerate(tstats.items()):
        ax_hist = axes[row_idx, 0]
        ax_qq = axes[row_idx, 1]
        ax_hist.hist(vals, bins=30, color="#3C6E71", edgecolor="black", alpha=0.8)
        ax_hist.set_title(f"{label}: histogram")
        ax_hist.set_xlabel("Studentized statistic")
        ax_hist.set_ylabel("Count")
        stats.probplot(vals, dist="norm", plot=ax_qq)
        ax_qq.set_title(f"{label}: QQ plot")
    save_fig(path)


def plot_length_vs_blocksize(df: pd.DataFrame, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6, 4))
    for learner, sub in df.groupby("learner"):
        sub_sorted = sub.sort_values("block_size")
        ax.plot(sub_sorted["block_size"], sub_sorted["avg_len"], marker="o", label=learner)
    ax.set_xlabel("Block size")
    ax.set_ylabel("Average CI length")
    if df["learner"].nunique() > 1:
        ax.legend(frameon=False)
    save_fig(path)


def plot_paths(pi: np.ndarray, q_cum: np.ndarray, path: Path) -> None:
    fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
    axes[0].plot(pi, color="#2F3E46")
    axes[0].set_ylabel(r"$\pi_t$")
    axes[0].set_title("Executed propensity path")
    axes[1].plot(q_cum, color="#52796F")
    axes[1].set_ylabel(r"$\sum_{s\leq t} \widehat\phi_s^2$")
    axes[1].set_xlabel("t")
    axes[1].set_title("Cumulative realized quadratic variation")
    save_fig(path)


def plot_wald_diagnostic_curve(
    df: pd.DataFrame,
    path: Path,
    x_col: str,
    y_col: str,
    hue_col: str,
    y_ref: float | None = None,
    y_label: str | None = None,
    title: str | None = None,
) -> None:
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for label, sub in df.groupby(hue_col, dropna=False):
        sub_sorted = sub.sort_values(x_col)
        ax.plot(sub_sorted[x_col], sub_sorted[y_col], marker="o", label=str(label))
    if y_ref is not None:
        ax.axhline(y_ref, color="black", linestyle="--", linewidth=1.5, alpha=0.8)
    ax.set_xlabel(str(x_col))
    ax.set_ylabel(y_label if y_label is not None else str(y_col))
    if title is not None:
        ax.set_title(title)
    if df[hue_col].nunique(dropna=False) > 1:
        ax.legend(frameon=False)
    save_fig(path)


def plot_power_curve(
    df: pd.DataFrame,
    path: Path,
    x_col: str = "tau",
    y_col: str = "reject_rate",
    hue_col: str = "method",
    title: str | None = None,
) -> None:
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for label, sub in df.groupby(hue_col, dropna=False):
        sub_sorted = sub.sort_values(x_col)
        ax.plot(sub_sorted[x_col], sub_sorted[y_col], marker="o", label=str(label))
    ax.set_xlabel(r"Effect size $\tau$")
    ax.set_ylabel("Reject rate (power / type I)")
    if title is not None:
        ax.set_title(title)
    if df[hue_col].nunique(dropna=False) > 1:
        ax.legend(frameon=False)
    save_fig(path)


def plot_precision_frontier(
    df: pd.DataFrame,
    path: Path,
    title: str | None = None,
    method_col: str = "method",
    len_col: str = "avg_len",
) -> None:
    df = df.copy()
    df = df.sort_values(len_col, ascending=True)
    methods = [str(m) for m in df[method_col].tolist()]
    values = df[len_col].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(7.0, 4))
    ax.bar(np.arange(len(methods)), values, color="#4C4C4C", edgecolor="black", alpha=0.85)
    ax.set_xticks(np.arange(len(methods)))
    ax.set_xticklabels(methods, rotation=25, ha="right")
    ax.set_ylabel("Average CI length")
    if title is not None:
        ax.set_title(title)
    save_fig(path)


def plot_regime_conditional_coverage(
    df: pd.DataFrame,
    path: Path,
    nominal: float = 0.95,
    method_col: str = "method",
    regime_col: str = "regime",
    cov_col: str = "coverage",
    count_col: str = "count",
) -> None:
    regimes = [str(r) for r in sorted(df[regime_col].unique())]
    methods = [str(m) for m in df[method_col].unique()]
    x = np.arange(len(regimes), dtype=float)
    width = 0.8 / max(len(methods), 1)

    fig, ax = plt.subplots(figsize=(6.8, 4))
    for j, method in enumerate(methods):
        sub = df[df[method_col] == method].set_index(regime_col).reindex(regimes)
        coverage = sub[cov_col].to_numpy(dtype=float)
        count = sub[count_col].to_numpy(dtype=float)
        se = np.sqrt(np.maximum(coverage * (1.0 - coverage), 0.0) / np.maximum(count, 1.0))
        err = 1.96 * se
        offset = (j - (len(methods) - 1) / 2.0) * width
        ax.bar(
            x + offset,
            coverage,
            width=width,
            yerr=err,
            capsize=3,
            color="#3C6E71" if j == 0 else "#4C4C4C",
            edgecolor="black",
            alpha=0.85,
            label=method,
        )

    ax.axhline(nominal, color="black", linestyle="--", linewidth=1.5, alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(regimes)
    ax.set_xlabel("Regime")
    ax.set_ylabel("Conditional coverage")
    if len(methods) > 1:
        ax.legend(frameon=False)
    save_fig(path)


# ## Design Generators and Monte Carlo Runners (A, B, C1, C2)

# In[ ]:


def make_blocks(indices: np.ndarray, block_size: int) -> list[np.ndarray]:
    return [indices[i : i + block_size] for i in range(0, indices.size, block_size)]


def sample_correlated_normal(rng: np.random.Generator, n: int, d: int, rho: float) -> np.ndarray:
    idx = np.arange(d)
    cov = rho ** np.abs(np.subtract.outer(idx, idx))
    chol = np.linalg.cholesky(cov)
    z = rng.normal(size=(n, d))
    return z @ chol.T


def design_d_m0(x: np.ndarray) -> np.ndarray:
    x1, x2, x3, x4 = x[:, 0], x[:, 1], x[:, 2], x[:, 3]
    return 0.8 * x1 + 0.5 * x2**2 - 0.5 * np.cos(x3) + 0.25 * x4


def design_d_tau(x: np.ndarray) -> np.ndarray:
    x1, x2, x3, x4, x5 = x[:, 0], x[:, 1], x[:, 2], x[:, 3], x[:, 4]
    return 0.5 * x1 + 0.5 * np.sin(x2) + 0.25 * (x3 > 0) - 0.25 * x4 * x5


def design_d_sigma(x: np.ndarray) -> np.ndarray:
    x1 = x[:, 0]
    return 1.0 + 0.5 * np.abs(x1)


def design_e_m0(x: np.ndarray) -> np.ndarray:
    x1, x2, x3, x4, x5, x6 = x[:, 0], x[:, 1], x[:, 2], x[:, 3], x[:, 4], x[:, 5]
    return 0.6 * x1 - 0.3 * x2 + 0.2 * x3**2 + 0.25 * np.sin(x4) + 0.1 * x5 * x6


def design_e_tau(x: np.ndarray) -> np.ndarray:
    x1, x2, x3, x4, x5 = x[:, 0], x[:, 1], x[:, 2], x[:, 3], x[:, 4]
    return 0.4 * x1 + 0.3 * (x2 > 0) - 0.2 * x3 * x4 + 0.15 * np.sin(x5)


def approx_theta_design_d(cfg: DesignDConfig, n_draws: int = 20000, seed: int = 0) -> float:
    rng = np.random.default_rng(seed)
    x = sample_correlated_normal(rng, n_draws, cfg.d, cfg.rho)
    return float(np.mean(design_d_tau(x)))


def approx_theta_design_e(cfg: DesignEConfig, n_draws: int = 20000, seed: int = 0) -> float:
    rng = np.random.default_rng(seed)
    x = sample_correlated_normal(rng, n_draws, cfg.d, cfg.rho)
    return float(np.mean(design_e_tau(x)))


def expit(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-x))


def policy_eps_greedy(tau_hat: np.ndarray, eps: float) -> np.ndarray:
    return np.where(tau_hat >= 0.0, 1.0 - eps, eps)


def policy_softmax(tau_hat: np.ndarray, temp: float) -> np.ndarray:
    return expit(tau_hat / temp)


def clip_pi(pi: np.ndarray, low: float, high: float) -> np.ndarray:
    return np.minimum(high, np.maximum(low, pi))


def simulate_design_a_replication(
    n: int,
    cfg: DesignAConfig,
    rng: ReplicationRng,
    alpha: float,
) -> dict:
    n0 = cfg.n0
    if n <= n0:
        raise ValueError("n must exceed n0 for Design A.")

    a_burn = rng.rng.binomial(1, cfg.pi_burn, size=n0)
    eps1_burn = rng.rng.normal(0.0, cfg.sigma1, size=n0)
    eps0_burn = rng.rng.normal(0.0, cfg.sigma0, size=n0)
    y_burn = np.where(a_burn == 1, cfg.tau + eps1_burn, eps0_burn)

    pi_burn = cfg.pi_burn
    psi_burn = a_burn * y_burn / pi_burn - (1 - a_burn) * y_burn / (1 - pi_burn)
    tau_hat_burn = float(np.mean(psi_burn))

    regime_high = tau_hat_burn >= 0.0
    pi_star = cfg.pi_high if regime_high else cfg.pi_low

    n_main = n - n0
    a_main = rng.rng.binomial(1, pi_star, size=n_main)
    eps1_main = rng.rng.normal(0.0, cfg.sigma1, size=n_main)
    eps0_main = rng.rng.normal(0.0, cfg.sigma0, size=n_main)
    y_main = np.where(a_main == 1, cfg.tau + eps1_main, eps0_main)

    a = np.concatenate([a_burn, a_main])
    y = np.concatenate([y_burn, y_main])
    pi = np.full(n, pi_star, dtype=float)
    pi[:n0] = cfg.pi_burn

    idx = np.arange(n0, n)
    n_eff = idx.size
    phi = phi_score(y[idx], a[idx], pi[idx], np.zeros(n_eff), np.zeros(n_eff))
    sn = compute_sn_ci(phi, alpha=alpha)

    fixed_v = 31.25
    fixed = fixed_variance_ci(sn.theta_hat, fixed_v, n_eff=n_eff, alpha=alpha)

    regime_v = regime_variance_design_a(regime_high)
    regime_fixed = fixed_variance_ci(sn.theta_hat, regime_v, n_eff=n_eff, alpha=alpha)

    return {
        "n": n,
        "n_eff": n_eff,
        "regime": "high" if regime_high else "low",
        "phi": phi,
        "sn": sn,
        "fixed": fixed,
        "regime_fixed": regime_fixed,
    }


def run_design_a(global_cfg: GlobalConfig, cfg: DesignAConfig) -> pd.DataFrame:
    n_list = cfg.n_list(global_cfg)
    R = global_cfg.resolve_R()

    def one_rep(n: int, rep: int) -> list[dict]:
        rng = make_replication_rng(global_cfg.seed, cfg.name, n, rep)
        out = simulate_design_a_replication(n, cfg, rng, alpha=global_cfg.alpha)
        rows = []
        theta_true = cfg.tau
        for method, res in [
            ("SN", out["sn"]),
            ("Fixed-V", out["fixed"]),
            ("Regime-Fixed", out["regime_fixed"]),
        ]:
            rows.append(
                {
                    "design": cfg.name,
                    "n": out["n"],
                    "n_eff": out["n_eff"],
                    "rep": rep,
                    "method": method,
                    "theta_hat": res.theta_hat,
                    "vhat": out["sn"].vhat,
                    "se": res.se,
                    "ci_lower": res.ci_lower,
                    "ci_upper": res.ci_upper,
                    "q_hat": out["sn"].vhat * out["n_eff"],
                    "tstat": studentized_statistic(res.theta_hat, theta_true, res.se),
                    "regime": out["regime"],
                }
            )
        return rows

    rows: list[dict] = []
    for n in n_list:
        results = Parallel(n_jobs=global_cfg.n_jobs)(delayed(one_rep)(n, rep) for rep in range(R))
        for rep_rows in results:
            rows.extend(rep_rows)
    return pd.DataFrame(rows)


def run_design_b(global_cfg: GlobalConfig, cfg: DesignBConfig) -> pd.DataFrame:
    n_list = cfg.n_list(global_cfg)
    R = global_cfg.resolve_R()

    def one_rep(n: int, rep: int) -> list[dict]:
        rng = make_replication_rng(global_cfg.seed, cfg.name, n, rep)
        a = rng.rng.binomial(1, cfg.pi, size=n)
        eps1 = rng.rng.normal(0.0, cfg.sigma1, size=n)
        eps0 = rng.rng.normal(0.0, cfg.sigma0, size=n)
        y = np.where(a == 1, cfg.tau + eps1, eps0)
        pi = np.full(n, cfg.pi)
        phi = phi_score(y, a, pi, np.zeros(n), np.zeros(n))
        sn = compute_sn_ci(phi, alpha=global_cfg.alpha)
        fixed_v = cfg.sigma1**2 / cfg.pi + cfg.sigma0**2 / (1.0 - cfg.pi)
        fixed = fixed_variance_ci(sn.theta_hat, fixed_v, n_eff=n, alpha=global_cfg.alpha)
        rows = []
        theta_true = cfg.tau
        for method, res in [("SN", sn), ("Fixed-V", fixed)]:
            rows.append(
                {
                    "design": cfg.name,
                    "n": n,
                    "n_eff": n,
                    "rep": rep,
                    "method": method,
                    "theta_hat": res.theta_hat,
                    "vhat": sn.vhat,
                    "se": res.se,
                    "ci_lower": res.ci_lower,
                    "ci_upper": res.ci_upper,
                    "q_hat": sn.vhat * n,
                    "tstat": studentized_statistic(res.theta_hat, theta_true, res.se),
                }
            )
        return rows

    rows: list[dict] = []
    for n in n_list:
        results = Parallel(n_jobs=global_cfg.n_jobs)(delayed(one_rep)(n, rep) for rep in range(R))
        for rep_rows in results:
            rows.extend(rep_rows)
    return pd.DataFrame(rows)


def _reject_two_sided(ci_lower: float, ci_upper: float, null: float = 0.0) -> bool:
    return bool((ci_lower > null) or (ci_upper < null))


def simulate_design_c1_replication(n: int, cfg: DesignC1Config, tau: float, rng: ReplicationRng) -> dict:
    """Leakage / predictability stress test (C1).

    This follows the spirit of the paper's warning: predictable forward fitting yields a martingale score,
    while a nuisance that is fit *using future outcomes* can break the MDS and the fixed-horizon guarantee.

    DGP:
    - X_t in R^p i.i.d N(0, I_p)
    - m0(x) nonlinear; tau(x) = tau0 + delta*sin(x1) so ATE=tau0
    - Heavy-tailed errors (t with df>2, scaled to unit variance)
    - Adaptive executed propensity: pi_t = eps + (1-2eps)*expit(lam * (m1_hat(x)-m0_hat(x))),
      where m_hat is fit predictably by forward blocks.

    Methods:
    - Predictable nuisance: low-dimensional feature map + ridge, fit on past blocks.
    - Leaky nuisance: richer feature map fit on the full sample (uses future outcomes; not predictable).
    """
    tau0 = float(tau)
    if cfg.K < 2:
        raise ValueError("Design C1 requires K>=2 forward blocks.")
    if cfg.p < 3:
        raise ValueError("Design C1 uses features of x1/x2/x3; set p>=3.")
    if cfg.df <= 2:
        raise ValueError("Design C1 requires df>2 for finite variance.")

    def t_error(rng_: np.random.Generator, df: int, size: int) -> np.ndarray:
        scale = math.sqrt((df - 2.0) / df)
        return rng_.standard_t(df, size=size) * scale

    def m0_star(x_: np.ndarray) -> np.ndarray:
        x1 = x_[:, 0]
        x2 = x_[:, 1]
        x3 = x_[:, 2]
        return 0.5 * x1 + 0.25 * x2**2 - 0.25 + 0.5 * np.sin(x3)

    def tau_x(x_: np.ndarray, tau0_: float, delta_: float) -> np.ndarray:
        return tau0_ + delta_ * np.sin(x_[:, 0])

    def build_features_predictable(x_: np.ndarray) -> np.ndarray:
        n_ = x_.shape[0]
        x1 = x_[:, 0]
        x2 = x_[:, 1]
        x3 = x_[:, 2]
        cols = [
            np.ones(n_),
            x1,
            x2**2,
            np.sin(x3),
            np.sin(x1),
        ]
        return np.column_stack(cols)

    def build_features_leaky(x_: np.ndarray) -> np.ndarray:
        n_, p_ = x_.shape
        cols: list[np.ndarray] = [np.ones(n_)]
        cols.extend([x_[:, j] for j in range(p_)])
        cols.extend([x_[:, j] ** 2 for j in range(p_)])
        cols.extend([x_[:, j] ** 3 for j in range(p_)])
        cols.append(np.sin(x_[:, 0]))
        cols.append(np.sin(x_[:, 1]))
        cols.append(np.sin(x_[:, 2]))
        max_inter = min(p_, 10)
        for i in range(max_inter):
            for j in range(i + 1, max_inter):
                cols.append(x_[:, i] * x_[:, j])
        return np.column_stack(cols)

    def ridge_fit(phi: np.ndarray, y_: np.ndarray, ridge: float) -> np.ndarray:
        if y_.size == 0:
            return np.zeros(phi.shape[1])
        xtx = phi.T @ phi
        d_ = xtx.shape[0]
        xtx.flat[:: d_ + 1] += ridge
        return np.linalg.solve(xtx, phi.T @ y_)

    x = rng.rng.normal(0.0, 1.0, size=(n, cfg.p))
    m0_true = m0_star(x)
    tau_true = tau_x(x, tau0_=tau0, delta_=float(cfg.delta))
    m1_true = m0_true + tau_true

    eps0 = t_error(rng.rng, df=int(cfg.df), size=n)
    eps1 = t_error(rng.rng, df=int(cfg.df), size=n)

    pi = np.empty(n, dtype=float)
    a = np.empty(n, dtype=int)
    y = np.empty(n, dtype=float)
    mhat0_pred = np.zeros(n, dtype=float)
    mhat1_pred = np.zeros(n, dtype=float)

    phi_pred = build_features_predictable(x)
    phi_leaky = build_features_leaky(x)

    blocks = np.array_split(np.arange(n), int(cfg.K))
    burn_idx = blocks[0]
    pi[burn_idx] = 0.5
    a[burn_idx] = rng.rng.binomial(1, pi[burn_idx])
    y[burn_idx] = np.where(
        a[burn_idx] == 1,
        m1_true[burn_idx] + eps1[burn_idx],
        m0_true[burn_idx] + eps0[burn_idx],
    )

    for k in range(1, int(cfg.K)):
        past_idx = np.concatenate(blocks[:k])
        cur_idx = blocks[k]

        past_phi = phi_pred[past_idx]
        past_y = y[past_idx]
        past_a = a[past_idx]

        beta0 = ridge_fit(past_phi[past_a == 0], past_y[past_a == 0], float(cfg.ridge_pred))
        beta1 = ridge_fit(past_phi[past_a == 1], past_y[past_a == 1], float(cfg.ridge_pred))

        mhat0_pred[cur_idx] = phi_pred[cur_idx] @ beta0
        mhat1_pred[cur_idx] = phi_pred[cur_idx] @ beta1

        logits = float(cfg.lam) * (mhat1_pred[cur_idx] - mhat0_pred[cur_idx])
        pi[cur_idx] = float(cfg.eps) + (1.0 - 2.0 * float(cfg.eps)) * expit(logits)
        a[cur_idx] = rng.rng.binomial(1, pi[cur_idx])
        y[cur_idx] = np.where(
            a[cur_idx] == 1,
            m1_true[cur_idx] + eps1[cur_idx],
            m0_true[cur_idx] + eps0[cur_idx],
        )

    # Leaky nuisance: full-sample rich-feature ridge fits (uses future outcomes).
    full_beta0 = ridge_fit(phi_leaky[a == 0], y[a == 0], float(cfg.ridge_leaky))
    full_beta1 = ridge_fit(phi_leaky[a == 1], y[a == 1], float(cfg.ridge_leaky))
    mhat0_leaky = phi_leaky @ full_beta0
    mhat1_leaky = phi_leaky @ full_beta1

    scored_idx = np.concatenate(blocks[1:])
    n_eff = int(scored_idx.size)

    return {
        "a": a,
        "y": y,
        "pi": pi,
        "mhat0_pred": mhat0_pred,
        "mhat1_pred": mhat1_pred,
        "mhat0_leaky": mhat0_leaky,
        "mhat1_leaky": mhat1_leaky,
        "scored_idx": scored_idx,
        "n_eff": n_eff,
        "tau": tau0,
    }


def run_design_c1(global_cfg: GlobalConfig, cfg: DesignC1Config) -> pd.DataFrame:
    n_list = cfg.n_list(global_cfg)
    R = global_cfg.resolve_R()

    def one_rep(n: int, rep: int, tau: float) -> list[dict]:
        rng = make_replication_rng(global_cfg.seed, cfg.name, n, rep, extra=f"tau={tau:g}")
        data = simulate_design_c1_replication(n, cfg, tau=tau, rng=rng)
        idx = data["scored_idx"]
        y = data["y"][idx]
        a = data["a"][idx]
        pi = data["pi"][idx]
        n_eff = int(data["n_eff"])

        m0_pred = data["mhat0_pred"][idx]
        m1_pred = data["mhat1_pred"][idx]
        m0_leaky = data["mhat0_leaky"][idx]
        m1_leaky = data["mhat1_leaky"][idx]

        theta_true = float(data["tau"])
        rows = []
        for method, m0, m1 in [
            ("SN-AIPW-Predictable", m0_pred, m1_pred),
            ("SN-AIPW-LeakyFull", m0_leaky, m1_leaky),
            ("SN-IPW", np.zeros_like(m0_pred), np.zeros_like(m1_pred)),
        ]:
            phi = phi_score(y, a, pi, m0, m1)
            sn = compute_sn_ci(phi, alpha=global_cfg.alpha)
            rows.append(
                {
                    "design": cfg.name,
                    "n": n,
                    "n_eff": n_eff,
                    "rep": rep,
                    "tau": theta_true,
                    "theta_true": theta_true,
                    "method": method,
                    "theta_hat": sn.theta_hat,
                    "vhat": sn.vhat,
                    "se": sn.se,
                    "ci_lower": sn.ci_lower,
                    "ci_upper": sn.ci_upper,
                    "q_hat": sn.vhat * n_eff,
                    "tstat": studentized_statistic(sn.theta_hat, theta_true, sn.se),
                    "reject": _reject_two_sided(sn.ci_lower, sn.ci_upper, null=0.0),
                }
            )
        return rows

    rows: list[dict] = []
    for n in n_list:
        for tau in cfg.tau_grid:
            results = Parallel(n_jobs=global_cfg.n_jobs)(delayed(one_rep)(n, rep, float(tau)) for rep in range(R))
            for rep_rows in results:
                rows.extend(rep_rows)
    return pd.DataFrame(rows)


def _design_c2_f0(x: np.ndarray) -> np.ndarray:
    return x[:, 0] + x[:, 1]


def simulate_design_c2_replication(n: int, cfg: DesignC2Config, tau: float, rng: ReplicationRng) -> dict:
    x = sample_correlated_normal(rng.rng, n, cfg.d, cfg.rho)
    m0_true = _design_c2_f0(x)
    m1_true = m0_true + float(tau)

    pi = np.full(n, cfg.pi)
    a = rng.rng.binomial(1, cfg.pi, size=n)
    eps0 = rng.rng.normal(0.0, cfg.sigma, size=n)
    eps1 = rng.rng.normal(0.0, cfg.sigma, size=n)
    y = np.where(a == 1, m1_true + eps1, m0_true + eps0)

    blocks = [b.astype(int) for b in np.array_split(np.arange(n), cfg.K)]
    scored_idx = np.concatenate(blocks[1:])
    n_eff = int(scored_idx.size)

    mhat0_well = np.zeros(n)
    mhat1_well = np.zeros(n)
    mhat0_miss = np.zeros(n)
    mhat1_miss = np.zeros(n)

    for k in range(1, len(blocks)):
        train_idx = np.concatenate(blocks[:k])
        test_idx = blocks[k]

        spec_well = LearnerSpec(name="ols", random_state=rng.next_seed())
        model0, model1 = fit_arm_models(x[train_idx], a[train_idx], y[train_idx], spec_well)
        mhat0_well[test_idx] = model0.predict(x[test_idx])
        mhat1_well[test_idx] = model1.predict(x[test_idx])

        x_miss_train = x[train_idx][:, [0]]
        x_miss_test = x[test_idx][:, [0]]
        spec_miss = LearnerSpec(name="ols", random_state=rng.next_seed())
        model0_m, model1_m = fit_arm_models(x_miss_train, a[train_idx], y[train_idx], spec_miss)
        mhat0_miss[test_idx] = model0_m.predict(x_miss_test)
        mhat1_miss[test_idx] = model1_m.predict(x_miss_test)

    return {
        "x": x,
        "a": a,
        "y": y,
        "pi": pi,
        "m0_true": m0_true,
        "m1_true": m1_true,
        "mhat0_well": mhat0_well,
        "mhat1_well": mhat1_well,
        "mhat0_miss": mhat0_miss,
        "mhat1_miss": mhat1_miss,
        "scored_idx": scored_idx,
        "n_eff": n_eff,
        "tau": float(tau),
    }


def run_design_c2(global_cfg: GlobalConfig, cfg: DesignC2Config) -> pd.DataFrame:
    n_list = cfg.n_list(global_cfg)
    R = global_cfg.resolve_R()

    def one_rep(n: int, rep: int, tau: float) -> list[dict]:
        rng = make_replication_rng(global_cfg.seed, cfg.name, n, rep, extra=f"tau={tau:g}")
        data = simulate_design_c2_replication(n, cfg, tau=tau, rng=rng)
        idx = data["scored_idx"]
        y = data["y"][idx]
        a = data["a"][idx]
        pi = data["pi"][idx]
        n_eff = int(data["n_eff"])

        theta_true = float(data["tau"])
        m0_oracle = data["m0_true"][idx]
        m1_oracle = data["m1_true"][idx]
        m0_well = data["mhat0_well"][idx]
        m1_well = data["mhat1_well"][idx]
        m0_miss = data["mhat0_miss"][idx]
        m1_miss = data["mhat1_miss"][idx]

        rows = []
        for method, m0, m1 in [
            ("SN-AIPW-Oracle", m0_oracle, m1_oracle),
            ("SN-AIPW-WellSpec", m0_well, m1_well),
            ("SN-AIPW-Misspec", m0_miss, m1_miss),
            ("SN-IPW", np.zeros_like(m0_oracle), np.zeros_like(m1_oracle)),
        ]:
            phi = phi_score(y, a, pi, m0, m1)
            sn = compute_sn_ci(phi, alpha=global_cfg.alpha)
            rows.append(
                {
                    "design": cfg.name,
                    "n": n,
                    "n_eff": n_eff,
                    "rep": rep,
                    "tau": theta_true,
                    "theta_true": theta_true,
                    "method": method,
                    "theta_hat": sn.theta_hat,
                    "vhat": sn.vhat,
                    "se": sn.se,
                    "ci_lower": sn.ci_lower,
                    "ci_upper": sn.ci_upper,
                    "q_hat": sn.vhat * n_eff,
                    "tstat": studentized_statistic(sn.theta_hat, theta_true, sn.se),
                    "reject": _reject_two_sided(sn.ci_lower, sn.ci_upper, null=0.0),
                }
            )
        return rows

    rows: list[dict] = []
    for n in n_list:
        for tau in cfg.tau_grid:
            results = Parallel(n_jobs=global_cfg.n_jobs)(delayed(one_rep)(n, rep, float(tau)) for rep in range(R))
            for rep_rows in results:
                rows.extend(rep_rows)
    return pd.DataFrame(rows)


# ## Design Generators and Monte Carlo Runners (D, E, F)

# In[ ]:


def simulate_design_d_replication(
    n: int,
    cfg: DesignDConfig,
    policy: str,
    rng: ReplicationRng,
) -> dict:
    n0 = cfg.n0
    x = sample_correlated_normal(rng.rng, n, cfg.d, cfg.rho)
    m0_true = design_d_m0(x)
    tau_true = design_d_tau(x)
    m1_true = m0_true + tau_true
    sigma = design_d_sigma(x)
    eps0 = rng.rng.normal(0.0, sigma)
    eps1 = rng.rng.normal(0.0, sigma)

    a = np.zeros(n, dtype=int)
    y = np.zeros(n)
    pi = np.zeros(n)
    mhat0_pred = np.zeros(n)
    mhat1_pred = np.zeros(n)

    pi[:n0] = 0.5
    a[:n0] = rng.rng.binomial(1, pi[:n0])
    y[:n0] = np.where(a[:n0] == 1, m1_true[:n0] + eps1[:n0], m0_true[:n0] + eps0[:n0])

    scored_idx = np.arange(n0, n)
    blocks = make_blocks(scored_idx, cfg.block_size)

    for block in blocks:
        train_idx = np.arange(0, block[0])
        spec = LearnerSpec(name=cfg.learner, random_state=rng.next_seed())
        model0, model1 = fit_arm_models(x[train_idx], a[train_idx], y[train_idx], spec)
        mhat0_pred[block] = model0.predict(x[block])
        mhat1_pred[block] = model1.predict(x[block])
        tau_hat = mhat1_pred[block] - mhat0_pred[block]
        if policy == "eps-greedy":
            pi_block = policy_eps_greedy(tau_hat, cfg.eps_greedy)
        elif policy == "softmax":
            pi_block = policy_softmax(tau_hat, cfg.softmax_temp)
        else:
            raise ValueError(f"Unknown policy '{policy}'.")
        pi_block = clip_pi(pi_block, cfg.clip_low, cfg.clip_high)
        pi[block] = pi_block
        a[block] = rng.rng.binomial(1, pi_block)
        y[block] = np.where(a[block] == 1, m1_true[block] + eps1[block], m0_true[block] + eps0[block])

    spec_kfold = LearnerSpec(name=cfg.learner, random_state=rng.next_seed())
    mhat0_kfold, mhat1_kfold = kfold_predictions(x, a, y, spec_kfold, n_splits=5)

    return {
        "x": x,
        "a": a,
        "y": y,
        "pi": pi,
        "m0_true": m0_true,
        "m1_true": m1_true,
        "mhat0_pred": mhat0_pred,
        "mhat1_pred": mhat1_pred,
        "mhat0_kfold": mhat0_kfold,
        "mhat1_kfold": mhat1_kfold,
        "scored_idx": scored_idx,
    }


def run_design_d(global_cfg: GlobalConfig, cfg: DesignDConfig) -> pd.DataFrame:
    n_list = cfg.n_list(global_cfg)
    R = global_cfg.resolve_R()
    theta_true = approx_theta_design_d(cfg)

    def one_rep(n: int, rep: int, policy: str) -> list[dict]:
        rng = make_replication_rng(global_cfg.seed, cfg.name, n, rep, extra=policy)
        data = simulate_design_d_replication(n, cfg, policy, rng)
        idx = data["scored_idx"]
        y = data["y"][idx]
        a = data["a"][idx]
        pi = data["pi"][idx]
        n_eff = idx.size
        # Illustrate why Assumption "logging integrity" matters: if the analyst uses a
        # mis-logged propensity (e.g., assumes 0.5 for all units), IPW is generally biased.
        pi_assumed = np.full_like(pi, 0.5, dtype=float)

        m0_pred = data["mhat0_pred"][idx]
        m1_pred = data["mhat1_pred"][idx]
        m0_oracle = data["m0_true"][idx]
        m1_oracle = data["m1_true"][idx]
        m0_kfold = data["mhat0_kfold"][idx]
        m1_kfold = data["mhat1_kfold"][idx]

        rows = []
        for method, m0, m1, pi_used in [
            ("SN-AIPW", m0_pred, m1_pred, pi),
            ("SN-IPW", np.zeros_like(m0_pred), np.zeros_like(m1_pred), pi),
            ("SN-IPW-Assume0p5", np.zeros_like(m0_pred), np.zeros_like(m1_pred), pi_assumed),
            ("SN-Oracle", m0_oracle, m1_oracle, pi),
            ("Naive-iid-DML", m0_kfold, m1_kfold, pi),
        ]:
            phi = phi_score(y, a, pi_used, m0, m1)
            sn = compute_sn_ci(phi, alpha=global_cfg.alpha)
            rows.append(
                {
                    "design": cfg.name,
                    "n": n,
                    "n_eff": n_eff,
                    "rep": rep,
                    "policy": policy,
                    "method": method,
                    "theta_hat": sn.theta_hat,
                    "vhat": sn.vhat,
                    "se": sn.se,
                    "ci_lower": sn.ci_lower,
                    "ci_upper": sn.ci_upper,
                    "q_hat": sn.vhat * n_eff,
                    "tstat": studentized_statistic(sn.theta_hat, theta_true, sn.se),
                }
            )
        return rows

    rows: list[dict] = []
    policies = ["eps-greedy", "softmax"]
    for n in n_list:
        for policy in policies:
            results = Parallel(n_jobs=global_cfg.n_jobs)(
                delayed(one_rep)(n, rep, policy) for rep in range(R)
            )
            for rep_rows in results:
                rows.extend(rep_rows)
    return pd.DataFrame(rows)


def simulate_design_e_replication(
    n: int,
    cfg: DesignEConfig,
    learner: str,
    block_size: int,
    rng: ReplicationRng,
) -> dict:
    n0 = cfg.n0
    x = sample_correlated_normal(rng.rng, n, cfg.d, cfg.rho)
    m0_true = design_e_m0(x)
    tau_true = design_e_tau(x)
    m1_true = m0_true + tau_true

    eps0 = rng.rng.normal(0.0, 1.0, size=n)
    eps1 = rng.rng.normal(0.0, 1.0, size=n)

    a = np.zeros(n, dtype=int)
    y = np.zeros(n)
    pi = np.zeros(n)
    mhat0_pred = np.zeros(n)
    mhat1_pred = np.zeros(n)

    pi[:n0] = 0.5
    a[:n0] = rng.rng.binomial(1, pi[:n0])
    y[:n0] = np.where(a[:n0] == 1, m1_true[:n0] + eps1[:n0], m0_true[:n0] + eps0[:n0])

    scored_idx = np.arange(n0, n)
    blocks = make_blocks(scored_idx, block_size)

    for block in blocks:
        train_idx = np.arange(0, block[0])
        spec = LearnerSpec(name=learner, random_state=rng.next_seed())
        model0, model1 = fit_arm_models(x[train_idx], a[train_idx], y[train_idx], spec)
        mhat0_pred[block] = model0.predict(x[block])
        mhat1_pred[block] = model1.predict(x[block])
        tau_hat = mhat1_pred[block] - mhat0_pred[block]
        pi_block = clip_pi(expit(tau_hat), cfg.clip_low, cfg.clip_high)
        pi[block] = pi_block
        a[block] = rng.rng.binomial(1, pi_block)
        y[block] = np.where(a[block] == 1, m1_true[block] + eps1[block], m0_true[block] + eps0[block])

    return {
        "x": x,
        "a": a,
        "y": y,
        "pi": pi,
        "m0_true": m0_true,
        "m1_true": m1_true,
        "mhat0_pred": mhat0_pred,
        "mhat1_pred": mhat1_pred,
        "scored_idx": scored_idx,
    }


def run_design_e(global_cfg: GlobalConfig, cfg: DesignEConfig) -> pd.DataFrame:
    n_list = cfg.n_list(global_cfg)
    R = global_cfg.resolve_R()

    def one_rep(n: int, rep: int, learner: str, block_size: int) -> list[dict]:
        extra = f"{learner}-{block_size}"
        rng = make_replication_rng(global_cfg.seed, cfg.name, n, rep, extra=extra)
        data = simulate_design_e_replication(n, cfg, learner, block_size, rng)
        idx = data["scored_idx"]
        y = data["y"][idx]
        a = data["a"][idx]
        pi = data["pi"][idx]
        n_eff = idx.size

        m0_pred = data["mhat0_pred"][idx]
        m1_pred = data["mhat1_pred"][idx]
        m0_oracle = data["m0_true"][idx]
        m1_oracle = data["m1_true"][idx]

        rows = []
        for method, m0, m1 in [("SN-AIPW", m0_pred, m1_pred), ("SN-Oracle", m0_oracle, m1_oracle)]:
            phi = phi_score(y, a, pi, m0, m1)
            sn = compute_sn_ci(phi, alpha=global_cfg.alpha)
            rows.append(
                {
                    "design": cfg.name,
                    "n": n,
                    "n_eff": n_eff,
                    "rep": rep,
                    "learner": learner,
                    "block_size": block_size,
                    "method": method,
                    "theta_hat": sn.theta_hat,
                    "vhat": sn.vhat,
                    "se": sn.se,
                    "ci_lower": sn.ci_lower,
                    "ci_upper": sn.ci_upper,
                    "q_hat": sn.vhat * n_eff,
                }
            )
        return rows

    rows: list[dict] = []
    for n in n_list:
        for learner in cfg.learners:
            for block_size in cfg.block_sizes:
                results = Parallel(n_jobs=global_cfg.n_jobs)(
                    delayed(one_rep)(n, rep, learner, block_size) for rep in range(R)
                )
                for rep_rows in results:
                    rows.extend(rep_rows)
    return pd.DataFrame(rows)


def simulate_design_f_replication(
    n: int,
    cfg: DesignFConfig,
    rng: ReplicationRng,
    scenario: str,
    clip_low: float,
    heavy_df: int | None = None,
    log_mode: str | None = None,
) -> dict:
    n0 = cfg.n0
    x = sample_correlated_normal(rng.rng, n, cfg.d, cfg.rho)
    m0_true = design_d_m0(x)
    tau_true = design_d_tau(x)
    m1_true = m0_true + tau_true

    if heavy_df is None:
        eps0 = rng.rng.normal(0.0, 1.0, size=n)
        eps1 = rng.rng.normal(0.0, 1.0, size=n)
    else:
        scale = math.sqrt((heavy_df - 2.0) / heavy_df)
        eps0 = rng.rng.standard_t(heavy_df, size=n) * scale
        eps1 = rng.rng.standard_t(heavy_df, size=n) * scale

    a = np.zeros(n, dtype=int)
    y = np.zeros(n)
    pi_exec = np.zeros(n)
    mhat0_pred = np.zeros(n)
    mhat1_pred = np.zeros(n)

    pi_exec[:n0] = 0.5
    a[:n0] = rng.rng.binomial(1, pi_exec[:n0])
    y[:n0] = np.where(a[:n0] == 1, m1_true[:n0] + eps1[:n0], m0_true[:n0] + eps0[:n0])

    scored_idx = np.arange(n0, n)
    blocks = make_blocks(scored_idx, cfg.block_size)

    for block in blocks:
        train_idx = np.arange(0, block[0])
        spec = LearnerSpec(name=cfg.learner, random_state=rng.next_seed())
        model0, model1 = fit_arm_models(x[train_idx], a[train_idx], y[train_idx], spec)
        mhat0_pred[block] = model0.predict(x[block])
        mhat1_pred[block] = model1.predict(x[block])
        tau_hat = mhat1_pred[block] - mhat0_pred[block]
        pi_block = policy_softmax(tau_hat, cfg.softmax_temp)
        pi_block = clip_pi(pi_block, clip_low, 1.0 - clip_low)
        pi_exec[block] = pi_block
        a[block] = rng.rng.binomial(1, pi_block)
        y[block] = np.where(a[block] == 1, m1_true[block] + eps1[block], m0_true[block] + eps0[block])

    pi_logged = pi_exec.copy()
    if log_mode == "rounded":
        pi_logged = np.round(pi_exec, 1)
        pi_logged = clip_pi(pi_logged, 0.01, 0.99)
    elif log_mode == "lagged":
        pi_logged[1:] = pi_exec[:-1]
        pi_logged = clip_pi(pi_logged, 0.01, 0.99)
    elif log_mode == "preclip":
        raw_pi = policy_softmax(mhat1_pred - mhat0_pred, cfg.softmax_temp)
        pi_logged = clip_pi(raw_pi, 0.01, 0.99)

    spec_kfold = LearnerSpec(name=cfg.learner, random_state=rng.next_seed())
    mhat0_kfold, mhat1_kfold = kfold_predictions(x, a, y, spec_kfold, n_splits=5)

    return {
        "x": x,
        "a": a,
        "y": y,
        "pi_exec": pi_exec,
        "pi_logged": pi_logged,
        "m0_true": m0_true,
        "m1_true": m1_true,
        "mhat0_pred": mhat0_pred,
        "mhat1_pred": mhat1_pred,
        "mhat0_kfold": mhat0_kfold,
        "mhat1_kfold": mhat1_kfold,
        "scored_idx": scored_idx,
    }


def run_design_f(global_cfg: GlobalConfig, cfg: DesignFConfig) -> pd.DataFrame:
    n_list = cfg.n_list(global_cfg)
    R = global_cfg.resolve_R()
    rows: list[dict] = []

    def one_rep_f1(n: int, rep: int, clip_low: float) -> list[dict]:
        rng = make_replication_rng(global_cfg.seed, cfg.name, n, rep, extra=f"F1-{clip_low}")
        data = simulate_design_f_replication(n, cfg, rng, "F1", clip_low)
        idx = data["scored_idx"]
        y = data["y"][idx]
        a = data["a"][idx]
        pi = data["pi_exec"][idx]
        n_eff = idx.size
        m0_pred = data["mhat0_pred"][idx]
        m1_pred = data["mhat1_pred"][idx]
        m0_kfold = data["mhat0_kfold"][idx]
        m1_kfold = data["mhat1_kfold"][idx]

        rows_local = []
        for method, m0, m1 in [("SN-AIPW", m0_pred, m1_pred), ("Naive-iid-DML", m0_kfold, m1_kfold)]:
            phi = phi_score(y, a, pi, m0, m1)
            sn = compute_sn_ci(phi, alpha=global_cfg.alpha)
            rows_local.append(
                {
                    "design": cfg.name,
                    "panel": "F1",
                    "scenario": f"pmin={clip_low:.2f}",
                    "n": n,
                    "n_eff": n_eff,
                    "rep": rep,
                    "method": method,
                    "theta_hat": sn.theta_hat,
                    "vhat": sn.vhat,
                    "se": sn.se,
                    "ci_lower": sn.ci_lower,
                    "ci_upper": sn.ci_upper,
                    "q_hat": sn.vhat * n_eff,
                }
            )
        return rows_local

    def one_rep_f2(n: int, rep: int, mode: str) -> list[dict]:
        rng = make_replication_rng(global_cfg.seed, cfg.name, n, rep, extra=f"F2-{mode}")
        data = simulate_design_f_replication(n, cfg, rng, "F2", clip_low=0.05, log_mode=mode)
        idx = data["scored_idx"]
        y = data["y"][idx]
        a = data["a"][idx]
        pi = data["pi_logged"][idx]
        n_eff = idx.size
        m0_pred = data["mhat0_pred"][idx]
        m1_pred = data["mhat1_pred"][idx]
        phi = phi_score(y, a, pi, m0_pred, m1_pred)
        sn = compute_sn_ci(phi, alpha=global_cfg.alpha)
        return [
            {
                "design": cfg.name,
                "panel": "F2",
                "scenario": mode,
                "n": n,
                "n_eff": n_eff,
                "rep": rep,
                "method": "SN-AIPW",
                "theta_hat": sn.theta_hat,
                "vhat": sn.vhat,
                "se": sn.se,
                "ci_lower": sn.ci_lower,
                "ci_upper": sn.ci_upper,
                "q_hat": sn.vhat * n_eff,
            }
        ]

    def one_rep_f3(n: int, rep: int, df: int) -> list[dict]:
        rng = make_replication_rng(global_cfg.seed, cfg.name, n, rep, extra=f"F3-{df}")
        data = simulate_design_f_replication(n, cfg, rng, "F3", clip_low=0.05, heavy_df=df)
        idx = data["scored_idx"]
        y = data["y"][idx]
        a = data["a"][idx]
        pi = data["pi_exec"][idx]
        n_eff = idx.size
        m0_pred = data["mhat0_pred"][idx]
        m1_pred = data["mhat1_pred"][idx]

        rows_local = []
        phi = phi_score(y, a, pi, m0_pred, m1_pred)
        sn = compute_sn_ci(phi, alpha=global_cfg.alpha)
        rows_local.append(
            {
                "design": cfg.name,
                "panel": "F3",
                "scenario": f"$t_{{{df}}}$",
                "n": n,
                "n_eff": n_eff,
                "rep": rep,
                "method": "SN-AIPW",
                "theta_hat": sn.theta_hat,
                "vhat": sn.vhat,
                "se": sn.se,
                "ci_lower": sn.ci_lower,
                "ci_upper": sn.ci_upper,
                "q_hat": sn.vhat * n_eff,
            }
        )
        phi_win = winsorize(phi, 0.01, 0.99)
        sn_win = compute_sn_ci(phi_win, alpha=global_cfg.alpha)
        rows_local.append(
            {
                "design": cfg.name,
                "panel": "F3",
                "scenario": f"$t_{{{df}}}$",
                "n": n,
                "n_eff": n_eff,
                "rep": rep,
                "method": "SN-AIPW-Winsor",
                "theta_hat": sn_win.theta_hat,
                "vhat": sn_win.vhat,
                "se": sn_win.se,
                "ci_lower": sn_win.ci_lower,
                "ci_upper": sn_win.ci_upper,
                "q_hat": sn_win.vhat * n_eff,
            }
        )
        return rows_local

    for n in n_list:
        for clip_low in cfg.clip_lows:
            results = Parallel(n_jobs=global_cfg.n_jobs)(
                delayed(one_rep_f1)(n, rep, clip_low) for rep in range(R)
            )
            for rep_rows in results:
                rows.extend(rep_rows)
        for mode in ["rounded", "lagged", "preclip"]:
            results = Parallel(n_jobs=global_cfg.n_jobs)(
                delayed(one_rep_f2)(n, rep, mode) for rep in range(R)
            )
            for rep_rows in results:
                rows.extend(rep_rows)
        for df in cfg.heavy_df:
            results = Parallel(n_jobs=global_cfg.n_jobs)(
                delayed(one_rep_f3)(n, rep, df) for rep in range(R)
            )
            for rep_rows in results:
                rows.extend(rep_rows)

    return pd.DataFrame(rows)


# ## Artifact I/O, Unified Runner, Table/Figure Builders

# In[ ]:


def _method_slug(method: str) -> str:
    return method.replace(" ", "_")


def save_replications(df: pd.DataFrame, design: str, global_cfg: GlobalConfig, design_cfg) -> None:
    design_dir = global_cfg.artifacts_dir / design
    design_dir.mkdir(parents=True, exist_ok=True)
    meta_path = design_dir / "config.json"
    meta = {
        "global": asdict(global_cfg),
        "design": asdict(design_cfg),
    }
    meta_path.write_text(json.dumps(meta, indent=2, default=str), encoding="utf-8")

    for method, sub in df.groupby("method"):
        method_dir = design_dir / _method_slug(method)
        method_dir.mkdir(parents=True, exist_ok=True)
        sub.to_csv(method_dir / "replications.csv", index=False)


def load_design_replications(design: str, global_cfg: GlobalConfig) -> pd.DataFrame:
    design_dir = global_cfg.artifacts_dir / design
    if not design_dir.exists():
        raise FileNotFoundError(f"Missing artifacts for design {design} in {design_dir}.")
    frames = []
    for method_dir in design_dir.iterdir():
        if not method_dir.is_dir():
            continue
        path = method_dir / "replications.csv"
        if path.exists():
            frames.append(pd.read_csv(path))
    if not frames:
        raise FileNotFoundError(f"No replication files found under {design_dir}.")
    return pd.concat(frames, ignore_index=True)


def run_designs(
    designs: list[str],
    global_cfg: GlobalConfig,
    cfg_a: DesignAConfig | None = None,
    cfg_b: DesignBConfig | None = None,
    cfg_c1: DesignC1Config | None = None,
    cfg_c2: DesignC2Config | None = None,
    cfg_d: DesignDConfig | None = None,
) -> dict[str, pd.DataFrame]:
    cfg_a = cfg_a or DesignAConfig()
    cfg_b = cfg_b or DesignBConfig()
    cfg_c1 = cfg_c1 or DesignC1Config()
    cfg_c2 = cfg_c2 or DesignC2Config()
    cfg_d = cfg_d or DesignDConfig()

    expanded: list[str] = []
    for d in designs:
        if d == "C":
            expanded.extend(["C1", "C2"])
        else:
            expanded.append(d)
    seen: set[str] = set()
    designs = [d for d in expanded if not (d in seen or seen.add(d))]

    outputs: dict[str, pd.DataFrame] = {}
    for design in designs:
        if design == "A":
            df = run_design_a(global_cfg, cfg_a)
            save_replications(df, "A", global_cfg, cfg_a)
        elif design == "B":
            df = run_design_b(global_cfg, cfg_b)
            save_replications(df, "B", global_cfg, cfg_b)
        elif design == "C1":
            df = run_design_c1(global_cfg, cfg_c1)
            save_replications(df, "C1", global_cfg, cfg_c1)
        elif design == "C2":
            df = run_design_c2(global_cfg, cfg_c2)
            save_replications(df, "C2", global_cfg, cfg_c2)
        elif design == "D":
            df = run_design_d(global_cfg, cfg_d)
            save_replications(df, "D", global_cfg, cfg_d)
        else:
            raise ValueError(f"Unknown design '{design}'.")

        outputs[design] = df
        print(f"Completed design {design}: {len(df):,} rows")

    return outputs


def _save_simple_csv(path: Path, df: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def make_simple_tables(
    global_cfg: GlobalConfig,
    designs: list[str] | None = None,
    cfg_d: DesignDConfig | None = None,
) -> dict[str, pd.DataFrame]:
    """Build plain CSV summaries for selected designs (A, B, C1, C2, D)."""
    cfg_d = cfg_d or DesignDConfig()

    if designs is None:
        selected = {"A", "B", "C1", "C2", "D"}
    else:
        expanded: list[str] = []
        for d in designs:
            if d == "C":
                expanded.extend(["C1", "C2"])
            else:
                expanded.append(d)
        selected = set(expanded)

    out: dict[str, pd.DataFrame] = {}

    if "A" in selected:
        try:
            design_a = load_design_replications("A", global_cfg)
            theta_a = 0.0
            df_a = add_basic_metrics(design_a, theta_a)
            summary_a = (
                df_a.groupby(["n", "n_eff", "method"], dropna=False)
                .agg(coverage=("covered", "mean"), avg_len=("length", "mean"), n_rep=("covered", "size"))
                .reset_index()
            )
            summary_a["mcse"] = summary_a.apply(
                lambda row: math.sqrt(row["coverage"] * (1.0 - row["coverage"]) / row["n_rep"]), axis=1
            )
            cond_a = (
                df_a.groupby(["n", "n_eff", "method", "regime"], dropna=False)
                .agg(
                    coverage=("covered", "mean"),
                    avg_len=("length", "mean"),
                    count=("covered", "size"),
                )
                .reset_index()
            )
            cond_a["mcse"] = cond_a.apply(
                lambda row: math.sqrt(row["coverage"] * (1.0 - row["coverage"]) / row["count"]), axis=1
            )
            _save_simple_csv(global_cfg.tables_dir / "sim_designA_main.csv", summary_a)
            _save_simple_csv(global_cfg.tables_dir / "sim_designA_conditional.csv", cond_a)
            out["sim_designA_main"] = summary_a
            out["sim_designA_conditional"] = cond_a
            print("Wrote Design A CSV tables")
        except FileNotFoundError:
            pass

    if "B" in selected:
        try:
            design_b = load_design_replications("B", global_cfg)
            theta_b = 0.0
            df_b = add_basic_metrics(design_b, theta_b)
            summary_b = (
                df_b.groupby(["n", "n_eff", "method"], dropna=False)
                .agg(coverage=("covered", "mean"), avg_len=("length", "mean"), n_rep=("covered", "size"))
                .reset_index()
            )
            summary_b["mcse"] = summary_b.apply(
                lambda row: math.sqrt(row["coverage"] * (1.0 - row["coverage"]) / row["n_rep"]), axis=1
            )
            _save_simple_csv(global_cfg.tables_dir / "sim_designB.csv", summary_b)
            out["sim_designB"] = summary_b
            print("Wrote Design B CSV table")
        except FileNotFoundError:
            pass

    if "C1" in selected:
        try:
            c1 = load_design_replications("C1", global_cfg)
            df = c1.copy()
            df["covered"] = (df["ci_lower"] <= df["theta_true"]) & (df["theta_true"] <= df["ci_upper"])
            df["length"] = df["ci_upper"] - df["ci_lower"]
            df["reject"] = df.get("reject", (df["ci_lower"] > 0.0) | (df["ci_upper"] < 0.0))

            df_null = df[np.isclose(df["tau"], 0.0)].copy()
            summary = (
                df_null.groupby(["n", "n_eff", "method"], dropna=False)
                .agg(
                    coverage=("covered", "mean"),
                    avg_len=("length", "mean"),
                    reject_rate=("reject", "mean"),
                    n_rep=("covered", "size"),
                )
                .reset_index()
            )
            summary["mcse"] = summary.apply(
                lambda row: math.sqrt(row["coverage"] * (1.0 - row["coverage"]) / row["n_rep"]), axis=1
            )
            _save_simple_csv(global_cfg.tables_dir / "sim_designC1.csv", summary)
            out["sim_designC1"] = summary
            print("Wrote Design C1 CSV table")
        except FileNotFoundError:
            pass

    if "C2" in selected:
        try:
            c2 = load_design_replications("C2", global_cfg)
            df = c2.copy()
            df["covered"] = (df["ci_lower"] <= df["theta_true"]) & (df["theta_true"] <= df["ci_upper"])
            df["length"] = df["ci_upper"] - df["ci_lower"]
            df["bias"] = df["theta_hat"] - df["theta_true"]
            df["sq_error"] = df["bias"] ** 2

            df_null = df[np.isclose(df["tau"], 0.0)].copy()
            summary = (
                df_null.groupby(["n", "n_eff", "method"], dropna=False)
                .agg(
                    coverage=("covered", "mean"),
                    avg_len=("length", "mean"),
                    rmse=("sq_error", lambda x: float(np.sqrt(np.mean(x)))),
                    vbar=("vhat", "mean"),
                    n_rep=("covered", "size"),
                )
                .reset_index()
            )
            summary["mcse"] = summary.apply(
                lambda row: math.sqrt(row["coverage"] * (1.0 - row["coverage"]) / row["n_rep"]), axis=1
            )
            oracle = summary[summary["method"] == "SN-AIPW-Oracle"][["n", "n_eff", "avg_len", "vbar"]].rename(
                columns={"avg_len": "oracle_len", "vbar": "oracle_v"}
            )
            summary = summary.merge(oracle, on=["n", "n_eff"], how="left")
            summary["rel_len"] = summary["avg_len"] / summary["oracle_len"]
            summary["rel_vhat"] = summary["vbar"] / summary["oracle_v"]
            _save_simple_csv(global_cfg.tables_dir / "sim_designC2.csv", summary)
            out["sim_designC2"] = summary
            print("Wrote Design C2 CSV table")
        except FileNotFoundError:
            pass

    if "D" in selected:
        try:
            design_d = load_design_replications("D", global_cfg)
            theta_d = approx_theta_design_d(cfg_d)
            df_d = add_basic_metrics(design_d, theta_d)
            summary_d = (
                df_d.groupby(["n", "n_eff", "policy", "method"], dropna=False)
                .agg(
                    coverage=("covered", "mean"),
                    avg_len=("length", "mean"),
                    bias=("bias", "mean"),
                    n_rep=("covered", "size"),
                )
                .reset_index()
            )
            summary_d["mcse"] = summary_d.apply(
                lambda row: math.sqrt(row["coverage"] * (1.0 - row["coverage"]) / row["n_rep"]), axis=1
            )
            _save_simple_csv(global_cfg.tables_dir / "sim_designD.csv", summary_d)
            out["sim_designD"] = summary_d

            subset = design_d[
                (design_d["n"] == design_d["n"].max()) & (design_d["method"].isin(["SN-AIPW", "Naive-iid-DML"]))
            ]
            cond = conditional_coverage_by_decile(subset, theta_true=theta_d)
            _save_simple_csv(global_cfg.tables_dir / "sim_conditional_by_Q_decile.csv", cond)
            out["sim_conditional_by_Q_decile"] = cond
            print("Wrote Design D CSV tables")
        except FileNotFoundError:
            pass

    return out



def make_figures(
    global_cfg: GlobalConfig,
    designs: list[str] | None = None,
    cfg_d: DesignDConfig | None = None,
) -> None:
    cfg_d = cfg_d or DesignDConfig()

    if designs is None:
        selected = {"A", "B", "C1", "C2", "D"}
    else:
        expanded: list[str] = []
        for d in designs:
            if d == "C":
                expanded.extend(["C1", "C2"])
            else:
                expanded.append(d)
        selected = set(expanded)

    design_a = None
    if "A" in selected:
        try:
            design_a = load_design_replications("A", global_cfg)
            max_n = design_a["n"].max()
            vhat = design_a[(design_a["n"] == max_n) & (design_a["method"] == "SN")]["vhat"].to_numpy()
            plot_var_ratio_hist(vhat, global_cfg.figures_dir / "var_ratio_hist.pdf")
        except FileNotFoundError:
            design_a = None

    if "D" in selected:
        try:
            design_d = load_design_replications("D", global_cfg)
            theta_d = approx_theta_design_d(cfg_d)
            subset = design_d[
                (design_d["n"] == design_d["n"].max()) & (design_d["method"].isin(["SN-AIPW", "Naive-iid-DML"]))
            ]
            tstats = {}
            for method in ["SN-AIPW", "Naive-iid-DML"]:
                data = subset[subset["method"] == method]
                tstats[method] = (data["theta_hat"] - theta_d) / data["se"]
            plot_hist_qq(tstats, global_cfg.figures_dir / "designD_tstat_histqq.pdf")
        except FileNotFoundError:
            pass

    if "A" in selected and design_a is not None:
        try:
            df_a = add_basic_metrics(design_a, theta_true=0.0)
            max_n = int(df_a["n"].max())
            subset = df_a[(df_a["n"] == max_n) & (df_a["method"].isin(["SN", "Fixed-V", "Regime-Fixed"]))]
            cond = (
                subset.groupby(["method", "regime"], dropna=False)
                .agg(coverage=("covered", "mean"), count=("covered", "size"))
                .reset_index()
            )
            plot_regime_conditional_coverage(cond, global_cfg.figures_dir / "designA_conditional_coverage.pdf")
        except Exception:
            pass

    if "C1" in selected:
        try:
            c1 = load_design_replications("C1", global_cfg)
            c1 = c1.copy()
            c1["covered"] = (c1["ci_lower"] <= c1["theta_true"]) & (c1["theta_true"] <= c1["ci_upper"])
            c1["reject"] = c1.get("reject", (c1["ci_lower"] > 0.0) | (c1["ci_upper"] < 0.0))
            c1_null = c1[np.isclose(c1["tau"], 0.0)].copy()

            summary = (
                c1_null.groupby(["n", "method"], dropna=False)
                .agg(coverage=("covered", "mean"), reject_rate=("reject", "mean"), n_rep=("covered", "size"))
                .reset_index()
            )
            plot_wald_diagnostic_curve(
                summary,
                x_col="n",
                y_col="coverage",
                hue_col="method",
                y_ref=1.0 - global_cfg.alpha,
                path=global_cfg.figures_dir / "designC1_coverage_vs_n.pdf",
                y_label="Coverage",
                title="Design C1: leakage stress test (tau=0)",
            )

            max_n = int(c1["n"].max())
            subset = c1_null[(c1_null["n"] == max_n) & (c1_null["method"].isin(["SN-AIPW-Predictable", "SN-AIPW-LeakyFull"]))]
            tstats = {m: subset[subset["method"] == m]["tstat"].to_numpy() for m in subset["method"].unique()}
            plot_hist_qq(tstats, global_cfg.figures_dir / "designC1_tstat_histqq.pdf")

            power = (
                c1[(c1["n"] == max_n)]
                .groupby(["tau", "method"], dropna=False)
                .agg(reject_rate=("reject", "mean"))
                .reset_index()
            )
            plot_power_curve(
                power,
                path=global_cfg.figures_dir / "designC1_power_curve.pdf",
                x_col="tau",
                y_col="reject_rate",
                hue_col="method",
                title=f"Design C1: power/type-I vs tau (n={max_n})",
            )
        except FileNotFoundError:
            pass

    if "C2" in selected:
        try:
            c2 = load_design_replications("C2", global_cfg)
            c2 = c2.copy()
            c2["covered"] = (c2["ci_lower"] <= c2["theta_true"]) & (c2["theta_true"] <= c2["ci_upper"])
            c2["length"] = c2["ci_upper"] - c2["ci_lower"]
            c2["reject"] = c2.get("reject", (c2["ci_lower"] > 0.0) | (c2["ci_upper"] < 0.0))
            c2_null = c2[np.isclose(c2["tau"], 0.0)].copy()
            max_n = int(c2_null["n"].max())
            sub = c2_null[c2_null["n"] == max_n]
            precision = (
                sub.groupby(["method"], dropna=False)
                .agg(avg_len=("length", "mean"), coverage=("covered", "mean"))
                .reset_index()
            )
            plot_precision_frontier(
                precision,
                path=global_cfg.figures_dir / "designC2_precision_frontier.pdf",
                title=f"Design C2: precision frontier (tau=0, n={max_n})",
            )

            power = (
                c2[(c2["n"] == max_n)]
                .groupby(["tau", "method"], dropna=False)
                .agg(reject_rate=("reject", "mean"))
                .reset_index()
            )
            plot_power_curve(
                power,
                path=global_cfg.figures_dir / "designC2_power_curve.pdf",
                x_col="tau",
                y_col="reject_rate",
                hue_col="method",
                title=f"Design C2: power/type-I vs tau (n={max_n})",
            )
        except FileNotFoundError:
            pass


# ## Regime-Switching Studentization Experiment
# 
# This section inlines the standalone experiment from `simulate_regime_switching_studentization.py`.
# It can be run independently from Designs A/B/C1/C2/D.

# In[ ]:


LAST_REGIME_HIGH = None
E_ASSUMED = 0.5


def standard_normal_pdf(x: np.ndarray) -> np.ndarray:
    return np.exp(-0.5 * x**2) / math.sqrt(2.0 * math.pi)


def normal_pdf(x: np.ndarray, sd: float) -> np.ndarray:
    return np.exp(-0.5 * (x / sd) ** 2) / (sd * math.sqrt(2.0 * math.pi))


def norm_cdf(x: np.ndarray | float) -> np.ndarray | float:
    if "scipy_norm" in globals():
        return scipy_norm.cdf(x)
    return NormalDist().cdf(x)


def simulate_regime_one_replication(
    n: int,
    n0: int,
    tau: float,
    sigma0: float,
    sigma1: float,
    e_low: float,
    e_high: float,
    e_burn: float,
    rng: np.random.Generator,
):
    global LAST_REGIME_HIGH

    if n <= n0:
        raise ValueError("n must exceed n0 for the regime-switching design.")

    a_burn = rng.binomial(1, e_burn, size=n0)
    eps1_burn = rng.normal(0.0, sigma1, size=n0)
    eps0_burn = rng.normal(0.0, sigma0, size=n0)
    y_burn = np.where(a_burn == 1, tau + eps1_burn, eps0_burn)

    psi_burn = a_burn * y_burn / e_burn - (1 - a_burn) * y_burn / (1 - e_burn)
    tau_hat_burn = float(np.mean(psi_burn))

    regime_high = tau_hat_burn >= 0.0
    e_star = e_high if regime_high else e_low
    LAST_REGIME_HIGH = regime_high

    n_main = n - n0
    a_main = rng.binomial(1, e_star, size=n_main)
    eps1_main = rng.normal(0.0, sigma1, size=n_main)
    eps0_main = rng.normal(0.0, sigma0, size=n_main)
    y_main = np.where(a_main == 1, tau + eps1_main, eps0_main)

    a = np.concatenate([a_burn, a_main])
    y = np.concatenate([y_burn, y_main])
    e = np.full(n, e_star, dtype=float)
    e[:n0] = e_burn

    psi = a * y / e - (1 - a) * y / (1 - e)
    tau_hat = psi.mean()

    xi = psi - tau_hat
    v_hat = np.sum(xi**2)
    se_sn = math.sqrt(v_hat) / n
    tstat_sn = (tau_hat - tau) / se_sn

    var_fixed = sigma1**2 / E_ASSUMED + sigma0**2 / (1.0 - E_ASSUMED)
    se_fixed = math.sqrt(var_fixed / n)
    tstat_fixed = (tau_hat - tau) / se_fixed

    return tau_hat, se_sn, se_fixed, tstat_sn, tstat_fixed


def run_regime_monte_carlo(
    n_list,
    R,
    seed,
    n0=50,
    tau=0.0,
    sigma0=1.0,
    sigma1=3.0,
    e_low=0.2,
    e_high=0.8,
    e_burn=0.5,
    alpha=0.05,
):
    results_rows = []
    tstats = {}
    z = float(norm_ppf(1.0 - alpha / 2.0))

    for idx, n in enumerate(n_list):
        rng = np.random.default_rng(seed + idx)

        tau_hats = np.empty(R)
        se_sn = np.empty(R)
        se_fixed = np.empty(R)
        tstat_sn = np.empty(R)
        tstat_fixed = np.empty(R)
        regime_high = np.empty(R, dtype=bool)
        vhat_over_n = np.empty(R)

        count_high = 0

        for r in range(R):
            tau_hat, se_SN, se_F, t_SN, t_F = simulate_regime_one_replication(
                n=n,
                n0=n0,
                tau=tau,
                sigma0=sigma0,
                sigma1=sigma1,
                e_low=e_low,
                e_high=e_high,
                e_burn=e_burn,
                rng=rng,
            )

            tau_hats[r] = tau_hat
            se_sn[r] = se_SN
            se_fixed[r] = se_F
            tstat_sn[r] = t_SN
            tstat_fixed[r] = t_F

            if LAST_REGIME_HIGH is None:
                raise RuntimeError("Regime indicator was not set.")
            regime_high[r] = LAST_REGIME_HIGH
            count_high += int(LAST_REGIME_HIGH)
            vhat_over_n[r] = (se_SN * n) ** 2 / n

        count_low = R - count_high
        freq_high = count_high / R
        freq_low = count_low / R
        mean_tau = tau_hats.mean()
        sd_tau = tau_hats.std(ddof=1)

        print(
            f"n={n}: regime high={freq_high:.3f}, low={freq_low:.3f}; "
            f"mean tau_hat={mean_tau:.4f}, sd={sd_tau:.4f}"
        )

        ci_lower_sn = tau_hats - z * se_sn
        ci_upper_sn = tau_hats + z * se_sn
        coverage_sn = np.mean((ci_lower_sn <= tau) & (tau <= ci_upper_sn))
        avg_length_sn = np.mean(ci_upper_sn - ci_lower_sn)
        reject_sn = np.mean(np.abs(tstat_sn) > z)

        ci_lower_fixed = tau_hats - z * se_fixed
        ci_upper_fixed = tau_hats + z * se_fixed
        coverage_fixed = np.mean((ci_lower_fixed <= tau) & (tau <= ci_upper_fixed))
        avg_length_fixed = np.mean(ci_upper_fixed - ci_lower_fixed)
        reject_fixed = np.mean(np.abs(tstat_fixed) > z)

        results_rows.append(
            {
                "n": n,
                "method": "SN",
                "coverage": coverage_sn,
                "avg_length": avg_length_sn,
                "reject_rate": reject_sn,
                "mean_tau_hat": mean_tau,
                "sd_tau_hat": sd_tau,
            }
        )
        results_rows.append(
            {
                "n": n,
                "method": "Fixed",
                "coverage": coverage_fixed,
                "avg_length": avg_length_fixed,
                "reject_rate": reject_fixed,
                "mean_tau_hat": mean_tau,
                "sd_tau_hat": sd_tau,
            }
        )

        tstats[n] = {
            "SN": tstat_sn,
            "Fixed": tstat_fixed,
            "tau_hat": tau_hats,
            "se_sn": se_sn,
            "se_fixed": se_fixed,
            "regime_high": regime_high,
            "vhat_over_n": vhat_over_n,
        }

    results_df = pd.DataFrame(results_rows)
    return results_df, tstats


def summarize_regime_unconditional(results_df: pd.DataFrame) -> pd.DataFrame:
    return results_df.sort_values(["n", "method"]).reset_index(drop=True)


def summarize_regime_conditional(tstats: dict, n_list: list[int], tau: float, alpha: float) -> pd.DataFrame:
    z = float(norm_ppf(1.0 - alpha / 2.0))
    rows = []

    for n in n_list:
        data = tstats[n]
        regime_high = data["regime_high"]
        tau_hat = data["tau_hat"]
        se_sn = data["se_sn"]
        se_fixed = data["se_fixed"]
        tstat_sn = data["SN"]
        tstat_fixed = data["Fixed"]

        for regime_name, mask in [("high", regime_high), ("low", ~regime_high)]:
            count = int(mask.sum())
            if count == 0:
                continue

            ci_lower_sn = tau_hat[mask] - z * se_sn[mask]
            ci_upper_sn = tau_hat[mask] + z * se_sn[mask]
            coverage_sn = np.mean((ci_lower_sn <= tau) & (tau <= ci_upper_sn))
            avg_length_sn = np.mean(ci_upper_sn - ci_lower_sn)
            reject_sn = np.mean(np.abs(tstat_sn[mask]) > z)

            rows.append(
                {
                    "n": n,
                    "method": "SN",
                    "regime": regime_name,
                    "coverage": coverage_sn,
                    "avg_length": avg_length_sn,
                    "reject_rate": reject_sn,
                    "count": count,
                }
            )

            ci_lower_fixed = tau_hat[mask] - z * se_fixed[mask]
            ci_upper_fixed = tau_hat[mask] + z * se_fixed[mask]
            coverage_fixed = np.mean((ci_lower_fixed <= tau) & (tau <= ci_upper_fixed))
            avg_length_fixed = np.mean(ci_upper_fixed - ci_lower_fixed)
            reject_fixed = np.mean(np.abs(tstat_fixed[mask]) > z)

            rows.append(
                {
                    "n": n,
                    "method": "Fixed",
                    "regime": regime_name,
                    "coverage": coverage_fixed,
                    "avg_length": avg_length_fixed,
                    "reject_rate": reject_fixed,
                    "count": count,
                }
            )

    results = pd.DataFrame(rows)
    return results.sort_values(["n", "method", "regime"]).reset_index(drop=True)


def plot_tstat_hist(tstats_vals: np.ndarray, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(tstats_vals, bins=50, density=True, alpha=0.8, color="#4C4C4C", edgecolor="black")
    x = np.linspace(-4.5, 4.5, 400)
    ax.plot(x, standard_normal_pdf(x), color="black", linewidth=2.0, label="N(0,1)")
    ax.set_xlabel("t-statistic")
    ax.set_ylabel("Density")
    ax.legend(frameon=False)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path)
    plt.close(fig)


def plot_vhat_over_n(vhat_over_n: np.ndarray, v_high: float, v_low: float, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(vhat_over_n, bins=40, density=True, alpha=0.8, color="#4C4C4C", edgecolor="black")
    ax.axvline(v_high, color="black", linestyle="--", linewidth=1.5, label="v_high")
    ax.axvline(v_low, color="black", linestyle=":", linewidth=1.5, label="v_low")
    ax.set_xlabel(r"$\widehat V / n$")
    ax.set_ylabel("Density")
    ax.legend(frameon=False)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path)
    plt.close(fig)


def plot_tstat_fixed_mixture(tstat_fixed: np.ndarray, sd_high: float, sd_low: float, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(tstat_fixed, bins=50, density=True, alpha=0.8, color="#4C4C4C", edgecolor="black")

    x = np.linspace(-6.0, 6.0, 500)
    ax.plot(x, standard_normal_pdf(x), color="black", linewidth=2.0, label="N(0,1)")
    mix_density = 0.5 * normal_pdf(x, sd_high) + 0.5 * normal_pdf(x, sd_low)
    ax.plot(x, mix_density, color="black", linewidth=2.0, linestyle="--", label="Mixture")
    ax.set_xlabel("t-statistic")
    ax.set_ylabel("Density")
    ax.legend(frameon=False)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path)
    plt.close(fig)



def run_regime_switching_experiment(
    n_list: list[int],
    R: int,
    seed: int,
    n0: int = 50,
    tau: float = 0.0,
    sigma0: float = 1.0,
    sigma1: float = 3.0,
    e_burn: float = 0.5,
    e_low: float = 0.2,
    e_high: float = 0.8,
    alpha: float = 0.05,
    out_dir: Path = Path("artifacts/regime_switching"),
) -> tuple[pd.DataFrame, pd.DataFrame]:
    out_dir.mkdir(parents=True, exist_ok=True)

    results_df, tstats = run_regime_monte_carlo(
        n_list=n_list,
        R=R,
        seed=seed,
        n0=n0,
        tau=tau,
        sigma0=sigma0,
        sigma1=sigma1,
        e_low=e_low,
        e_high=e_high,
        e_burn=e_burn,
        alpha=alpha,
    )

    results_df = summarize_regime_unconditional(results_df)
    results_df.to_csv(out_dir / "results_regime_switching.csv", index=False)

    cond_df = summarize_regime_conditional(tstats, n_list, tau=tau, alpha=alpha)
    cond_df.to_csv(out_dir / "results_regime_conditional.csv", index=False)

    v_high = sigma1**2 / e_high + sigma0**2 / (1.0 - e_high)
    v_low = sigma1**2 / e_low + sigma0**2 / (1.0 - e_low)
    var_fixed = sigma1**2 / E_ASSUMED + sigma0**2 / (1.0 - E_ASSUMED)
    sd_high = math.sqrt(v_high / var_fixed)
    sd_low = math.sqrt(v_low / var_fixed)

    fig_dir = out_dir / "figures"
    for n in n_list:
        plot_tstat_hist(tstats[n]["SN"], fig_dir / f"tstat_hist_SN_n{n}.png")
        plot_tstat_hist(tstats[n]["Fixed"], fig_dir / f"tstat_hist_Fixed_n{n}.png")
        plot_vhat_over_n(tstats[n]["vhat_over_n"], v_high, v_low, fig_dir / f"Vhat_over_n_hist_n{n}.png")
        plot_tstat_fixed_mixture(tstats[n]["Fixed"], sd_high, sd_low, fig_dir / f"tstat_fixed_mixture_overlay_n{n}.png")
    cond_df_out = cond_df.copy()
    cond_df_out["count"] = cond_df_out["count"].astype(int)
    cond_df_out.to_csv(out_dir / "results_regime_conditional.csv", index=False)

    z = float(norm_ppf(1.0 - alpha / 2.0))
    coverage_fixed_pred = 0.5 * (2 * norm_cdf(z / sd_high) - 1) + 0.5 * (2 * norm_cdf(z / sd_low) - 1)
    reject_fixed_pred = 1.0 - coverage_fixed_pred
    print(f"Regime experiment predicted fixed-method coverage: {coverage_fixed_pred:.4f}")
    print(f"Regime experiment predicted fixed-method reject rate: {reject_fixed_pred:.4f}")

    return results_df, cond_df


In [16]:
# -----------------------------
# Main controls (edit these)
# -----------------------------
SEED = 12345
ALPHA = 0.05
N_REPS = 1000                 # <- change this directly
N_LIST = [250, 500, 1000, 2000, 5000]         # horizons used unless a design override is set
N_JOBS = -1

DESIGNS_TO_RUN = ["A", "B", "C1", "C2", "D"]
RUN_SIMULATIONS = True
BUILD_SIMPLE_TABLES = True
BUILD_FIGURES = True
RUN_REGIME_EXPERIMENT = True

# Regime-switching experiment controls
REGIME_N_LIST = [250, 500]
REGIME_REPS = 300

# Global configuration
global_cfg = GlobalConfig(
    seed=SEED,
    alpha=ALPHA,
    n_list=tuple(N_LIST),
    n_reps=N_REPS,
    n_jobs=N_JOBS,
    artifacts_dir=Path("artifacts/sim"),
    tables_dir=Path("tables"),
    figures_dir=Path("figures"),
)

# Design configs
cfg_a = DesignAConfig()
cfg_b = DesignBConfig()
cfg_c1 = DesignC1Config()
cfg_c2 = DesignC2Config(n_list_override=(5000,))
cfg_d = DesignDConfig()

print(f"Designs: {DESIGNS_TO_RUN}")
print(f"N_REPS: {global_cfg.n_reps}")
print(f"N_LIST: {global_cfg.resolve_n_list()}")


Designs: ['A', 'B', 'C1', 'C2', 'D']
N_REPS: 1000
N_LIST: [250, 500, 1000, 2000, 5000]


In [17]:
outputs = {}
summary_tables = {}
regime_results = None
regime_conditional = None

if RUN_SIMULATIONS:
    outputs = run_designs(
        designs=DESIGNS_TO_RUN,
        global_cfg=global_cfg,
        cfg_a=cfg_a,
        cfg_b=cfg_b,
        cfg_c1=cfg_c1,
        cfg_c2=cfg_c2,
        cfg_d=cfg_d,
    )

if BUILD_SIMPLE_TABLES:
    summary_tables = make_simple_tables(
        global_cfg=global_cfg,
        designs=DESIGNS_TO_RUN,
        cfg_d=cfg_d,
    )
    print("\nSaved CSV summary tables:")
    for name in sorted(summary_tables):
        print(f"- {name} -> {global_cfg.tables_dir / (name + '.csv')}")

if BUILD_FIGURES:
    make_figures(global_cfg=global_cfg, designs=DESIGNS_TO_RUN, cfg_d=cfg_d)

if RUN_REGIME_EXPERIMENT:
    regime_results, regime_conditional = run_regime_switching_experiment(
        n_list=REGIME_N_LIST,
        R=REGIME_REPS,
        seed=SEED,
        out_dir=Path("artifacts/regime_switching"),
    )

print("Done.")


Completed design A: 15,000 rows
Completed design B: 10,000 rows
Completed design C1: 15,000 rows
Completed design C2: 16,000 rows
Completed design D: 50,000 rows
Wrote Design A CSV tables
Wrote Design B CSV table
Wrote Design C1 CSV table
Wrote Design C2 CSV table
Wrote Design D CSV tables

Saved CSV summary tables:
- sim_conditional_by_Q_decile -> tables/sim_conditional_by_Q_decile.csv
- sim_designA_conditional -> tables/sim_designA_conditional.csv
- sim_designA_main -> tables/sim_designA_main.csv
- sim_designB -> tables/sim_designB.csv
- sim_designC1 -> tables/sim_designC1.csv
- sim_designC2 -> tables/sim_designC2.csv
- sim_designD -> tables/sim_designD.csv
n=250: regime high=0.543, low=0.457; mean tau_hat=0.0410, sd=0.3402
n=500: regime high=0.480, low=0.520; mean tau_hat=0.0151, sd=0.2439
Regime experiment predicted fixed-method coverage: 0.8864
Regime experiment predicted fixed-method reject rate: 0.1136
Done.


In [18]:
if summary_tables:
    for name in sorted(summary_tables):
        print(f"\n=== {name} ===")
        display(summary_tables[name].head(20))

if regime_results is not None:
    print("\n=== regime_results ===")
    display(regime_results)

if regime_conditional is not None:
    print("\n=== regime_conditional ===")
    display(regime_conditional)



=== sim_conditional_by_Q_decile ===


,method,q_decile,coverage,avg_len,n_rep,q_min,q_max,mcse
0,Naive-iid-DML,0,0.967949,0.315819,156,74889.074784,170385.720045,0.014102
1,Naive-iid-DML,1,0.945355,0.335389,183,170596.315996,180383.006500,0.016801
2,Naive-iid-DML,2,0.965000,0.343490,200,180393.386350,187922.655185,0.012995
3,Naive-iid-DML,3,0.938389,0.350544,211,187929.007063,195860.728328,0.016553
4,Naive-iid-DML,4,0.959459,0.359178,222,195885.859828,208746.618420,0.013237
5,Naive-iid-DML,5,0.937500,0.377814,176,208877.460659,243590.907141,0.018246
6,Naive-iid-DML,6,0.956284,0.410309,183,243872.486275,277535.716775,0.015114
7,Naive-iid-DML,7,0.960000,0.431291,200,277640.221033,305432.587129,0.013856
8,Naive-iid-DML,8,0.968468,0.452733,222,305466.432166,337587.453991,0.011728
9,Naive-iid-DML,9,0.935223,0.485300,247,337842.809019,471966.547528,0.015661



=== sim_designA_conditional ===


,n,n_eff,method,regime,coverage,avg_len,count,mcse
0,250,200,Fixed-V,high,0.997925,1.549488,482,0.002073
1,250,200,Fixed-V,low,0.893822,1.549488,518,0.013536
2,250,200,Regime-Fixed,high,0.962656,1.117351,482,0.008636
3,250,200,Regime-Fixed,low,0.938224,1.885033,518,0.010578
4,250,200,SN,high,0.960581,1.119288,482,0.008863
5,250,200,SN,low,0.949807,1.865049,518,0.009593
6,500,450,Fixed-V,high,0.993939,1.032992,495,0.003488
7,500,450,Fixed-V,low,0.885149,1.032992,505,0.014188
8,500,450,Regime-Fixed,high,0.961616,0.744901,495,0.008635
9,500,450,Regime-Fixed,low,0.938614,1.256689,505,0.010682



=== sim_designA_main ===


...

,n,n_eff,policy,method,coverage,avg_len,bias,n_rep,mcse
0,250,150,eps-greedy,Naive-iid-DML,0.944,1.883275,0.018856,1000,0.007271
1,250,150,eps-greedy,SN-AIPW,0.944,1.905635,0.019263,1000,0.007271
2,250,150,eps-greedy,SN-IPW,0.934,2.083195,0.003284,1000,0.007851
3,250,150,eps-greedy,SN-IPW-Assume0p5,0.433,1.295614,0.702739,1000,0.015669
4,250,150,eps-greedy,SN-Oracle,0.946,1.516145,0.009290,1000,0.007147
5,250,150,softmax,Naive-iid-DML,0.940,1.933794,0.012640,1000,0.007510
6,250,150,softmax,SN-AIPW,0.948,1.952726,0.012015,1000,0.007021
7,250,150,softmax,SN-IPW,0.916,2.114638,0.002138,1000,0.008772
8,250,150,softmax,SN-IPW-Assume0p5,0.395,1.297660,0.730648,1000,0.015459
9,250,150,softmax,SN-Oracle,0.953,1.509163,-0.002688,1000,0.006693



=== regime_results ===


,n,method,coverage,avg_length,reject_rate,mean_tau_hat,sd_tau_hat
0,250,Fixed,0.900000,1.108723,0.100000,0.040952,0.340173
1,250,SN,0.950000,1.277677,0.050000,0.040952,0.340173
2,500,Fixed,0.886667,0.783986,0.113333,0.015123,0.243942
3,500,SN,0.933333,0.937949,0.066667,0.015123,0.243942



=== regime_conditional ===


,n,method,regime,coverage,avg_length,reject_rate,count
0,250,Fixed,high,0.950920,1.108723,0.049080,163
1,250,Fixed,low,0.839416,1.108723,0.160584,137
2,250,SN,high,0.944785,1.023291,0.055215,163
3,250,SN,low,0.956204,1.580340,0.043796,137
4,500,Fixed,high,0.958333,0.783986,0.041667,144
5,500,Fixed,low,0.820513,0.783986,0.179487,156
6,500,SN,high,0.937500,0.717307,0.062500,144
7,500,SN,low,0.929487,1.141620,0.070513,156
